# ETL de datos Refinitiv para panel S&P 500 (mensual, 2015–2025)

**Objetivo general:** construir un panel mensual firma–mes con información de mercado y fundamentals obtenida en Refinitiv (Eikon/RDP), listo para la etapa econométrica.

**Insumos principales:**
- Universo de firmas: `empresas_tickers_test.csv`.
- (Opcional) Archivos Excel de bonos corporativos en la carpeta definida en configuración.

**Productos principales:**
- `data/clean/panel_master.parquet`
- `data/clean/data_dictionary.csv`
- `outputs/qa/qa_report.csv` (o equivalente)

**Consideraciones operativas:**
- Se requiere `EIKON_APP_KEY` como variable de entorno.
- Algunas descargas son intensivas y pueden enfrentar límites de tasa (HTTP 429); el notebook incorpora pausas y reintentos.



## 2) Importaciones


In [ ]:
# ======================
# imports + inicialización de refinitiv
# ======================

from __future__ import annotations

# librerías estándar
import os
import time
import logging
from pathlib import Path
from typing import Iterable, Optional

# librerías de terceros
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# refinitiv / eikon
import eikon as ek

# cargar variables de entorno desde .env
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)

# leer clave de eikon desde variables de entorno
EIKON_APP_KEY = os.environ.get("EIKON_APP_KEY")

# inicializar conexión con refinitiv
ek.set_app_key(EIKON_APP_KEY)

Using key prefix: 00f40e04


## 3) Configuración


In [ ]:
# ======================
# config
# ======================

# horizonte temporal del panel mensual
START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

# root del proyecto
PROJECT_ROOT = Path.cwd().resolve().parent

# estructura de datos dentro del repo
DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "inputs"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
QA_DIR = OUTPUT_DIR / "qa"
LOG_DIR = OUTPUT_DIR / "logs"

# crear carpetas si no existen
for p in (INPUT_DIR, RAW_DIR, CLEAN_DIR, QA_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ======================
# inputs
# ======================

# universo base de empresas
UNIVERSE_FILE = INPUT_DIR / "empresas_tickers_test.csv"

# export consolidado de bonos desde refinitiv / workspace
BONDS_DIR = INPUT_DIR / "new_bonds"
BOND_BATCH_FILES = [
    BONDS_DIR / "first_batch.xlsx",
    BONDS_DIR / "second_batch.xlsx",
    BONDS_DIR / "third_batch.xlsx",
]

# ======================
# outputs
# ======================

OUT = {
    "universe_firms": CLEAN_DIR / "universe_firms.parquet",
    "firms_metadata": CLEAN_DIR / "firms_metadata.parquet",
    "bonds_metadata": CLEAN_DIR / "bonds_metadata.parquet",
    "equity_prices_daily": RAW_DIR / "equity_prices_daily.parquet",
    "equity_returns_monthly": CLEAN_DIR / "equity_returns_monthly.parquet",
    "market_prices_daily": RAW_DIR / "market_prices_daily.parquet",
    "market_returns_monthly": CLEAN_DIR / "market_returns_monthly.parquet",
    "bonds_universe_full": CLEAN_DIR / "bonds_universe_full.parquet",
    "oas_spreads_monthly_bond": RAW_DIR / "oas_spreads_monthly_bond.parquet",
    "cds_spreads_daily": RAW_DIR / "cds_spreads_daily.parquet",
    "panel_master": CLEAN_DIR / "panel_master.parquet",
    "data_dictionary": CLEAN_DIR / "data_dictionary.csv",
    "qa_report": QA_DIR / "qa_report.csv",
}

# validar disponibilidad de la clave de eikon
EIKON_APP_KEY = os.environ.get("EIKON_APP_KEY")
assert EIKON_APP_KEY, (
    "Falta la variable de entorno EIKON_APP_KEY. "
    "Ejemplo (Windows PowerShell): $env:EIKON_APP_KEY='TU_KEY'"
)

# parámetros operativos para requests
REQUEST_PAUSE_SEC = 0.25
RETRY_MAX = 5
RETRY_BACKOFF_SEC = 2.0

## 4) Registro (logging) y funciones auxiliares


In [ ]:
# ======================
# logging + helpers
# ======================

def _build_logger(name: str = "etl") -> logging.Logger:
    logger = logging.getLogger(name)

    # evitar duplicar handlers si la celda se reejecuta
    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        "%Y-%m-%d %H:%M:%S",
    )

    # handler de consola
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    # handler de archivo
    fh = logging.FileHandler(LOG_DIR / "etl.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    return logger


logger = _build_logger("etl")


def ensure_datetime(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Fuerza `df[col]` a datetime naive, sin alterar el resto de las columnas.
    """
    if col not in df.columns:
        raise KeyError(f"Columna '{col}' no existe en df.")

    out = df.copy()
    out[col] = pd.to_datetime(out[col], errors="coerce")

    # si la columna viniera con zona horaria, la convierte a naive
    if getattr(out[col].dt, "tz", None) is not None:
        out[col] = out[col].dt.tz_localize(None)

    return out


def safe_write_parquet(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Escribe un parquet y registra en el log el destino y el tamaño del output.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=index)
    logger.info(f"Output -> {path.as_posix()} | rows={len(df):,} cols={df.shape[1]}")


def sleep_request(pause_sec: float = REQUEST_PAUSE_SEC) -> None:
    # pausa breve entre requests para evitar saturar la api
    time.sleep(pause_sec)

## 5) Conexión a Refinitiv


In [5]:
# ======================
# REFINTIV CONNECTION
# ======================

ek.set_app_key(EIKON_APP_KEY)
logger.info("Refinitiv/Eikon: APP_KEY seteada.")


2026-04-01 03:15:46 | INFO | Refinitiv/Eikon: APP_KEY seteada.
2026-04-01 03:15:46,895 P[4908] [MainThread 19948] Refinitiv/Eikon: APP_KEY seteada.


### Construcción del universo de firmas a partir del S&P 500

En esta sección se construye el universo de emisores utilizado en el análisis a partir de la composición del índice S&P 500, obtenida directamente desde Refinitiv. Este enfoque reemplaza la selección manual inicial y permite asegurar trazabilidad, replicabilidad y consistencia con las fuentes de datos del proyecto.

#### Objetivo

Definir un universo amplio y sistemático de firmas elegibles, compatible con el pipeline existente y basado exclusivamente en información descargada desde Refinitiv.

#### Estrategia de construcción

Se consulta la cadena del S&P 500 en Refinitiv y se recuperan, para cada constituyente, tanto identificadores de mercado como variables de clasificación sectorial. A partir de esta descarga se construye un dataset estructurado que contiene:

- Identificadores principales:
  - `Issuer`
  - `Ticker`
  - `RIC`
  - `ISIN`
  - `CUSIP`
  - `PermID`

- Información de clasificación:
  - País de headquarters
  - Sector y subsectores según GICS

Luego de la descarga, se realiza una limpieza básica de strings, se eliminan observaciones sin identificadores clave y se deduplica el universo a nivel de ticker.

#### Output

El resultado se guarda en dos formatos:
- CSV (`empresas_tickers_test.csv`), para mantener compatibilidad con el flujo original del notebook.
- Parquet (`universe_firms.parquet`), para uso eficiente en etapas posteriores.

Este bloque constituye el punto de entrada del universo de firmas sobre el cual se construyen el resto de las variables del panel.

In [ ]:
# ======================
# universo s&p 500 -> refinitiv
# genera empresas_tickers_test.csv ampliado
# ======================

SPX_CHAIN = "0#.SPX"

SPX_FIELDS = [
    "TR.CommonName",
    "TR.TickerSymbol",
    "TR.RIC",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.HeadquartersCountry",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustryName",
]


def clean_str_series(s):
    # limpieza estándar de strings: strip + normalización de missing
    return (
        s.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


# descarga del universo del s&p 500 desde refinitiv
spx, err = ek.get_data([SPX_CHAIN], SPX_FIELDS)

# log de warnings si existen
if err is not None and len(err):
    logger.warning(f"Warnings Refinitiv universo S&P 500: {err}")

# validación básica de descarga
if spx is None or spx.empty:
    raise RuntimeError("No se pudo descargar el universo S&P 500 desde Refinitiv.")

# normalización de nombres de columnas
spx.columns = [str(c).strip() for c in spx.columns]

rename_map = {
    "Company Common Name": "Issuer",
    "Ticker Symbol": "Ticker",
    "RIC": "RIC",
    "ISIN": "ISIN",
    "CUSIP": "CUSIP",
    "Country of Headquarters": "HeadquartersCountry",
}

spx = spx.rename(columns={k: v for k, v in rename_map.items() if k in spx.columns})

# limpieza de columnas clave
for c in ["Issuer", "Ticker", "RIC", "ISIN", "CUSIP", "PermID"]:
    if c in spx.columns:
        spx[c] = clean_str_series(spx[c])

# limpieza mínima: eliminar observaciones sin identificadores básicos
spx = spx.dropna(subset=["Ticker", "Issuer"]).copy()
spx["Ticker"] = spx["Ticker"].str.upper().str.strip()

# deduplicación conservadora por ticker
spx = spx.drop_duplicates(subset=["Ticker"], keep="first").reset_index(drop=True)

# guardar outputs (csv + parquet)
spx.to_csv(UNIVERSE_FILE, index=False)
safe_write_parquet(spx, OUT["universe_firms"], index=False)

# logs informativos
logger.info(f"Universo S&P 500 guardado en: {UNIVERSE_FILE}")
logger.info(f"Firmas en universo ampliado: {len(spx):,}")

### Universo de bonos corporativos desde Workspace y restricción al universo S&P 500

En esta sección se construye el universo inicial de bonos corporativos a partir de exportaciones realizadas en Refinitiv Workspace y luego se restringe ese universo a los emisores incluidos en el universo de firmas definido previamente para el S&P 500.

#### Objetivo

Obtener una base consolidada de instrumentos de renta fija corporativa que sea escalable, trazable y consistente con el universo de emisores del proyecto, preservando los identificadores necesarios para las descargas posteriores de spreads y métricas de mercado.

#### Procedimiento

Primero se leen y concatenan los tres lotes de bonos exportados desde Workspace. Luego se normalizan nombres de columnas, formatos de texto e identificadores relevantes, y se convierten a tipo numérico o fecha aquellas variables que lo requieren, como cupón, vencimiento y monto emitido.

A continuación, se carga el universo de firmas del S&P 500 construido en una etapa previa y se realiza el emparejamiento principal a través del `Ticker`. Este criterio permite restringir el universo de bonos a instrumentos emitidos por firmas incluidas en el universo accionario de referencia, sin perder el identificador específico de cada bono (`bond_ric`), que será la clave operativa en las etapas posteriores.

Finalmente, se eliminan observaciones sin identificadores esenciales y se deduplica el universo a nivel de combinación `Ticker`–`bond_ric`.

#### Output

El resultado de esta etapa es `df_bonds_universe`, que se guarda en formatos parquet y CSV como universo consolidado de bonos elegibles para la descarga de OAS y otras variables de mercado.

#### Controles de calidad

Como verificación básica, se monitorean:
- la cantidad total de bonos leídos desde los archivos de origen;
- la cantidad de bonos que emparejan con el universo S&P 500;
- el número de `bond_ric` únicos;
- la cantidad de firmas efectivamente cubiertas;
- la cobertura del universo accionario inicial.

In [ ]:
# ==========================================================
# bonos | universo amplio desde workspace + filtro por s&p 500
# ==========================================================

# -------------------------------------------------
# 1) paths de entrada
# -------------------------------------------------
BONDS_DIR = INPUT_DIR / "new_bonds"

BOND_BATCH_FILES = [
    BONDS_DIR / "first_batch.xlsx",
    BONDS_DIR / "second_batch.xlsx",
    BONDS_DIR / "third_batch.xlsx",
]

# -------------------------------------------------
# 2) lectura y consolidación de exports de bonos
# -------------------------------------------------
dfs = []

for f in BOND_BATCH_FILES:
    if not f.exists():
        raise FileNotFoundError(f"No existe el archivo: {f}")

    logger.info(f"Leyendo archivo de bonos: {f.name}")
    df = pd.read_excel(f, dtype=str)
    df["source_file"] = f.name
    dfs.append(df)

bonds_raw = pd.concat(dfs, ignore_index=True)

logger.info(f"Bonos leídos desde Workspace: rows={len(bonds_raw):,} cols={bonds_raw.shape[1]}")

# -------------------------------------------------
# 3) limpieza y normalización
# -------------------------------------------------
def clean_str(s: pd.Series) -> pd.Series:
    # normalización estándar de strings para cruces y deduplicación
    return (
        s.astype("string")
        .str.upper()
        .str.strip()
        .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )

rename_map = {
    "Issuer": "Issuer",
    "Ticker": "Ticker",
    "Coupon": "Coupon",
    "Maturity": "Maturity",
    "ISIN": "ISIN",
    "Preferred RIC": "bond_ric",
    "Principal Currency": "Principal_Currency",
    "Amount Issued (USD)": "Amount_Issued_USD",
}

bonds = bonds_raw.rename(
    columns={k: v for k, v in rename_map.items() if k in bonds_raw.columns}
).copy()

for c in ["Issuer", "Ticker", "ISIN", "bond_ric", "Principal_Currency"]:
    if c in bonds.columns:
        bonds[c] = clean_str(bonds[c])

if "Coupon" in bonds.columns:
    bonds["Coupon"] = pd.to_numeric(bonds["Coupon"], errors="coerce")

if "Maturity" in bonds.columns:
    bonds["Maturity"] = pd.to_datetime(bonds["Maturity"], errors="coerce")

if "Amount_Issued_USD" in bonds.columns:
    bonds["Amount_Issued_USD"] = pd.to_numeric(bonds["Amount_Issued_USD"], errors="coerce")

# -------------------------------------------------
# 4) carga del universo s&p 500
# -------------------------------------------------
sp500 = pd.read_csv(UNIVERSE_FILE, dtype=str).copy()

sp500["Ticker"] = clean_str(sp500["Ticker"])
if "Issuer" in sp500.columns:
    sp500["Issuer"] = clean_str(sp500["Issuer"])

logger.info(f"Firmas en universo S&P 500: {len(sp500):,}")
logger.info(f"Tickers únicos en universo S&P 500: {sp500['Ticker'].nunique():,}")

# -------------------------------------------------
# 5) filtro principal: bonos emitidos por firmas del s&p 500
# -------------------------------------------------
bonds_sp500 = bonds.merge(
    sp500[["Ticker"]].drop_duplicates(),
    on="Ticker",
    how="inner",
    validate="m:1",
)

# -------------------------------------------------
# 6) limpieza final del universo de bonos
# -------------------------------------------------
req_cols = ["Issuer", "Ticker", "bond_ric"]
missing_req = [c for c in req_cols if c not in bonds_sp500.columns]
if missing_req:
    raise ValueError(f"Faltan columnas requeridas en bonds_sp500: {missing_req}")

df_bonds_universe = (
    bonds_sp500
    .dropna(subset=["Ticker", "bond_ric"])
    .drop_duplicates(subset=["Ticker", "bond_ric"])
    .reset_index(drop=True)
)

# -------------------------------------------------
# 7) controles básicos de calidad
# -------------------------------------------------
coverage = df_bonds_universe["Ticker"].nunique() / sp500["Ticker"].nunique()

logger.info(f"Bonos totales descargados: {len(bonds):,}")
logger.info(f"Bonos que emparejan con S&P 500: {len(df_bonds_universe):,}")
logger.info(f"RICs únicos de bonos: {df_bonds_universe['bond_ric'].nunique():,}")
logger.info(f"Empresas cubiertas por el universo de bonos: {df_bonds_universe['Ticker'].nunique():,}")
logger.info(f"Cobertura del universo de firmas: {coverage:.2%}")

# -------------------------------------------------
# 8) guardado oficial
# -------------------------------------------------
out_parquet = OUT.get("bonds_universe_full", CLEAN_DIR / "bonds_universe_full.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(df_bonds_universe, out_parquet, index=False)
df_bonds_universe.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")
logger.info(f"Bonos S&P 500 guardados: {len(df_bonds_universe):,}")

### Descarga de metadatos de empresas desde Refinitiv

En esta sección se obtienen los metadatos fundamentales de las firmas incluidas en el universo definido previamente, utilizando la API de Refinitiv. Esta información complementa los identificadores básicos del universo y permite enriquecer el panel con variables estructurales y de clasificación.

#### Objetivo

Construir una base consistente de metadatos a nivel firma que incluya identificadores estandarizados (ISIN, CUSIP, PermID) y variables de clasificación sectorial (GICS), asegurando la máxima cobertura posible para el universo de emisores.

#### Procedimiento

A partir del archivo de universo (`empresas_tickers_test.csv`), se define el conjunto de instrumentos a consultar en Refinitiv. Cuando el identificador `RIC` no está disponible, se construye un proxy a partir del ticker mediante la asignación de sufijos de mercado (por ejemplo, `.O` para Nasdaq y `.N` para NYSE).

La descarga se realiza en batches para mitigar restricciones de la API, incorporando reintentos automáticos en caso de fallas o respuestas incompletas. Una vez obtenidos los datos, se normalizan nombres de columnas y formatos de identificadores.

#### Control de calidad

Se evalúa la cobertura de variables clave, en particular `ISIN` y `CUSIP`, identificando firmas con metadatos incompletos. Este diagnóstico es relevante para etapas posteriores de emparejamiento entre bases de datos.

#### Output

El resultado se guarda como `company_metadata` en formatos parquet y CSV dentro del directorio de datos crudos, constituyendo la base de metadatos de firmas para el resto del pipeline.

In [ ]:
# ======================
# empresas -> metadata refinitiv
# ======================

FIELDS = [
    "TR.CommonName",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.HeadquartersCountry",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustryName",
]

REQUIRED_META_COLS = ["ISIN", "CUSIP"]

# -------------------------------------------------
# helpers
# -------------------------------------------------

def ticker_to_ric(ticker: str):
    # asignación simple de sufijos de mercado para construir RICs
    ticker = ticker.strip().upper()

    nasdaq = {"AAPL","MSFT","GOOGL","META","CSCO","PEP","HON"}
    nyse = {
        "JPM","BAC","C","WFC","GS","MS","XOM","CVX","COP","OXY",
        "JNJ","PFE","MRK","ABBV","PG","KO","MCD","WMT",
        "GE","MMM","CAT","DUK","NEE","SO","ORCL"
    }

    if ticker in nasdaq:
        return f"{ticker}.O"

    if ticker in nyse:
        return f"{ticker}.N"

    return f"{ticker}.N"


def normalize_str(s):
    # normalización de strings: strip + manejo de missing
    out = s.astype("string").str.strip()
    out = out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return out


def fetch_metadata_retry(rics, fields, retries=3):
    # descarga con reintentos ante errores de refinitiv
    for attempt in range(retries):

        try:
            df, err = ek.get_data(rics, fields)

            if err is not None and len(err):
                logger.warning(f"Refinitiv error: {err}")

            if df is not None and not df.empty:
                return df

            else:
                logger.warning(f"Intento {attempt+1}: dataframe vacío")

        except Exception as e:
            logger.warning(f"Intento {attempt+1}/{retries} falló con excepción: {e}")

        sleep_request(2)

    raise RuntimeError("Refinitiv no respondió después de varios intentos")


def chunked(iterable, size):
    # divide lista en batches para requests
    for i in range(0, len(iterable), size):
        yield iterable[i:i + size]


# -------------------------------------------------
# 1) leer universo de firmas
# -------------------------------------------------

df_universe = pd.read_csv(UNIVERSE_FILE, dtype=str)

missing_cols = [c for c in ["Ticker", "Issuer"] if c not in df_universe.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas {missing_cols}")

df_universe = (
    df_universe
    .dropna(subset=["Ticker", "Issuer"])
    .reset_index(drop=True)
)

# usar ric directo si está disponible; fallback a construcción por ticker
if "RIC" in df_universe.columns:
    df_universe["RIC"] = normalize_str(df_universe["RIC"])
else:
    df_universe["RIC"] = pd.NA

df_universe["ric_for_download"] = df_universe["RIC"]

mask_missing_ric = df_universe["ric_for_download"].isna()
df_universe.loc[mask_missing_ric, "ric_for_download"] = (
    df_universe.loc[mask_missing_ric, "Ticker"]
    .astype(str)
    .str.strip()
    .apply(ticker_to_ric)
)

rics = (
    df_universe["ric_for_download"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

logger.info(f"RICs a consultar: {len(rics)}")

# -------------------------------------------------
# 2) descarga de metadata (batch)
# -------------------------------------------------

meta_frames = []

for i, batch in enumerate(chunked(rics, 10), start=1):

    logger.info(f"Batch {i} | instrumentos={len(batch)}")

    df = fetch_metadata_retry(batch, FIELDS)

    if df is not None and not df.empty:
        meta_frames.append(df)

    sleep_request(0.5)

if not meta_frames:
    raise RuntimeError("No se descargó metadata de ninguna empresa.")

meta = pd.concat(meta_frames, ignore_index=True)

logger.info(f"Metadata descargada: filas={len(meta)}")


# -------------------------------------------------
# 3) normalización de columnas
# -------------------------------------------------

meta.columns = [c.replace("TR.", "").strip() for c in meta.columns]

if "Instrument" in meta.columns:
    meta = meta.rename(columns={"Instrument": "ric"})
elif "RIC" in meta.columns:
    meta = meta.rename(columns={"RIC": "ric"})

if "ric" in meta.columns:
    meta["ric"] = normalize_str(meta["ric"])

for col in ["ISIN", "CUSIP"]:
    if col in meta.columns:
        meta[col] = normalize_str(meta[col])


# -------------------------------------------------
# 4) control de calidad
# -------------------------------------------------

bad_mask = pd.Series(False, index=meta.index)

for col in REQUIRED_META_COLS:
    if col not in meta.columns:
        logger.warning(f"Falta columna esperada: {col}")
        continue

    bad_mask |= meta[col].isna()

bad_rows = meta[bad_mask]

logger.info(f"Empresas con metadata incompleta: {len(bad_rows)}")


# -------------------------------------------------
# 5) export de resultados
# -------------------------------------------------

out_parquet = RAW_DIR / "company_metadata.parquet"
out_csv = RAW_DIR / "company_metadata.csv"

safe_write_parquet(meta, out_parquet, index=False)
meta.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv}")

### Descarga de precios diarios de equity desde Refinitiv

En esta sección se descargan las series de precios y volumen para las acciones de las firmas incluidas en el universo definido previamente. Estas series constituyen el insumo principal para la construcción de retornos y medidas de riesgo en etapas posteriores.

#### Objetivo

Obtener series diarias de precios de cierre (`CLOSE`) y volumen (`VOLUME`) para cada firma del universo, maximizando la cobertura efectiva y preservando consistencia en los identificadores utilizados.

#### Procedimiento

A partir del universo de firmas, se define para cada observación un conjunto de identificadores candidatos (`RIC`). Cuando el RIC no está disponible directamente, se construyen alternativas a partir del ticker mediante sufijos de mercado (por ejemplo, `.N` para NYSE y `.O` para Nasdaq).

La descarga se realiza de manera iterativa por firma, probando secuencialmente los candidatos disponibles hasta obtener una serie válida. Para cada RIC exitoso, se normalizan los nombres de columnas, se estandariza la variable de fecha y se asegura la presencia de las variables requeridas.

Se incorporan pausas entre requests para mitigar restricciones de la API y reducir la probabilidad de errores de tipo rate limit.

#### Control de calidad

Se monitorean:
- RICs para los cuales se obtiene información válida;
- RICs sin datos disponibles;
- rango temporal de cada serie;
- series con inicio tardío respecto al período de análisis.

Estas validaciones permiten detectar problemas de cobertura o disponibilidad de datos que puedan afectar las estimaciones posteriores.

#### Output

El resultado se consolida en un único dataset de frecuencia diaria y se guarda en formatos parquet y CSV como `equity_prices_daily`, constituyendo la base para la construcción de retornos de equity.

In [ ]:
# ======================
# equity -> precios diarios (close, volume)
# ======================

# -------------------------------------------------
# 1) carga del universo de empresas
# -------------------------------------------------
df_emp = (
    pd.read_csv(UNIVERSE_FILE, dtype=str)
    .dropna(subset=["Ticker"])
    .copy()
)

df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

# usar RIC si ya está disponible en el universo
if "RIC" in df_emp.columns:
    df_emp["RIC"] = df_emp["RIC"].astype("string").str.strip()
else:
    df_emp["RIC"] = pd.NA

logger.info(f"Empresas en universo: {len(df_emp):,}")
logger.info(f"Tickers únicos: {df_emp['Ticker'].nunique():,}")
logger.info(f"RICs disponibles: {df_emp['RIC'].notna().sum():,}")


def ric_candidates_row(row) -> list[str]:
    """
    Genera candidatos de RIC para la descarga.
    Prioriza el RIC existente; si no, construye a partir del ticker.
    """
    ric = row.get("RIC", pd.NA)
    ticker = str(row["Ticker"]).strip()

    if pd.notna(ric) and str(ric).strip() != "":
        return [str(ric).strip()]

    # si el ticker ya incluye sufijo, usarlo directamente
    if "." in ticker:
        return [ticker]

    # fallback: sufijos de mercado estándar
    return [f"{ticker}.N", f"{ticker}.O"]


# -------------------------------------------------
# 2) descarga iterativa de series
# -------------------------------------------------
all_frames = []
success = 0
fail = 0

for _, row in df_emp.iterrows():

    ticker = row["Ticker"]
    got_it = False

    for ric in ric_candidates_row(row):

        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE", "VOLUME"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            if ts is not None and not ts.empty:

                # normalización de formato
                ts = ts.reset_index().rename(columns={"Date": "date"})
                ts["date"] = pd.to_datetime(ts["date"], errors="coerce")

                # estandarización de nombres de columnas
                cols_upper = {c: str(c).upper() for c in ts.columns}
                ts = ts.rename(columns=cols_upper)

                if "DATE" in ts.columns and "date" not in ts.columns:
                    ts = ts.rename(columns={"DATE": "date"})

                if "date" not in ts.columns:
                    raise ValueError(f"No se encontró columna de fecha para {ric}")

                # eliminar timezone si existe
                if getattr(ts["date"].dt, "tz", None) is not None:
                    ts["date"] = ts["date"].dt.tz_localize(None)

                ts["ric"] = ric

                # asegurar columnas requeridas
                if "CLOSE" not in ts.columns:
                    ts["CLOSE"] = pd.NA
                if "VOLUME" not in ts.columns:
                    ts["VOLUME"] = pd.NA

                all_frames.append(ts[["date", "ric", "CLOSE", "VOLUME"]])

                dmin = ts["date"].min()
                dmax = ts["date"].max()
                logger.info(f"OK {ric:<12} | {dmin.date()} -> {dmax.date()}")

                success += 1
                got_it = True
                break

        except Exception as e:
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")

        # pausa entre intentos
        sleep_request(0.6)

    if not got_it:
        logger.warning(f"Sin datos para ticker: {ticker}")
        fail += 1


logger.info(f"RICs con datos: {success:,}")
logger.info(f"RICs sin datos: {fail:,}")


# -------------------------------------------------
# 3) consolidación final
# -------------------------------------------------
if all_frames:
    out = pd.concat(all_frames, ignore_index=True)
else:
    out = pd.DataFrame(columns=["date", "ric", "CLOSE", "VOLUME"])

out = out.sort_values(["ric", "date"]).reset_index(drop=True)


# -------------------------------------------------
# 4) export de resultados
# -------------------------------------------------
out_parquet = OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out, out_parquet, index=False)
out.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")
logger.info(f"Total filas: {len(out):,}")


# -------------------------------------------------
# 5) control de cobertura temporal
# -------------------------------------------------
if not out.empty:

    late = out.groupby("ric")["date"].min().reset_index()
    late = late[late["date"] > pd.Timestamp("2016-01-01")]

    if not late.empty:
        logger.warning("Series que arrancan tarde (min date > 2016-01-01):")
        logger.warning(late.sort_values("date").head(50).to_string(index=False))
    else:
        logger.info("Cobertura temporal OK según el umbral definido.")

else:
    logger.warning("El output de equity quedó vacío.")

### Construcción de retornos logarítmicos diarios de equity

En esta sección se transforman las series de precios diarios en retornos logarítmicos a nivel firma, que constituyen la base para la medición de riesgo y la construcción de variables explicativas en el análisis empírico.

#### Objetivo

Obtener una serie consistente de retornos diarios para cada activo, asegurando comparabilidad entre firmas y consistencia temporal dentro de cada serie.

#### Metodología

A partir de los precios de cierre diarios (`CLOSE`), se calcula el retorno logarítmico como:

$$
r_{i,t} = \ln(P_{i,t}) - \ln(P_{i,t-1})
$$

donde $P_{i,t}$ es el precio de cierre del activo $i$ en el día $t$.

El cálculo se realiza de forma independiente para cada instrumento (`ric`), respetando el orden temporal de las observaciones. Las primeras observaciones de cada serie, para las cuales no existe un precio previo, se descartan.

#### Control de calidad

Se monitorean:

- la cantidad total de observaciones de entrada;
- el número de activos (`ric`) disponibles;
- la cantidad de observaciones con retorno válido;
- la cobertura efectiva de retornos por activo.

#### Output

El resultado se guarda como `equity_returns_daily` en formatos parquet y CSV, y constituye el insumo directo para la construcción de medidas de riesgo sistemático e idiosincrático.

In [ ]:
# ======================
# equity -> retornos log diarios
# ======================

# si no está en memoria, se puede cargar desde parquet
# out = pd.read_parquet(OUT.get("equity_prices_daily", RAW_DIR / "equity_prices_daily.parquet"))

# -------------------------------------------------
# 1) preparación de datos
# -------------------------------------------------
out = (
    out
    .sort_values(["ric", "date"])
    .reset_index(drop=True)
    .copy()
)

# asegurar tipo numérico en precios
out["CLOSE"] = pd.to_numeric(out["CLOSE"], errors="coerce")

logger.info(f"Filas en precios (entrada): {len(out):,}")
logger.info(f"RICs únicos en precios: {out['ric'].nunique():,}")


# -------------------------------------------------
# 2) cálculo de retornos logarítmicos
# -------------------------------------------------
# r_it = log(P_t) - log(P_{t-1}) por activo
out["ret_log"] = (
    out
    .groupby("ric")["CLOSE"]
    .transform(lambda s: np.log(s).diff())
)

# eliminar observaciones sin retorno (primer dato de cada serie o missing)
out_ret = out.dropna(subset=["ret_log"]).copy()

logger.info(f"Filas con ret_log válido: {len(out_ret):,}")
logger.info(f"RICs con al menos un retorno: {out_ret['ric'].nunique():,}")


# -------------------------------------------------
# 3) export de resultados
# -------------------------------------------------
out_parquet = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(out_ret, out_parquet, index=False)
out_ret.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")

### Construcción del factor de mercado y emparejamiento con retornos de equity

En esta sección se construye la serie de retornos del mercado y se la integra con los retornos individuales de las firmas. Este paso permite definir el componente agregado necesario para la estimación de riesgo sistemático en el panel.

#### Objetivo

Obtener una medida consistente del retorno diario del mercado y emparejarla con los retornos de cada activo, generando una base conjunta para el análisis de exposición a shocks agregados.

#### Procedimiento

Se descarga una serie representativa del mercado accionario utilizando Refinitiv. Dado que distintos instrumentos pueden presentar problemas de disponibilidad, se define un conjunto de candidatos (índices y ETFs) y se selecciona automáticamente el primero con datos válidos.

A partir de los precios de cierre, se construye el retorno logarítmico del mercado:

$$
r^{mkt}_t = \ln(P_t) - \ln(P_{t-1})
$$

Luego, esta serie se empareja con los retornos de equity a nivel fecha, generando una base unificada que contiene:

- retorno del activo (`ret_log`);
- retorno del mercado (`mkt_ret_log`).

El merge se realiza por intersección temporal, asegurando consistencia entre ambas series.

#### Control de calidad

Se verifican:
- disponibilidad de la serie de mercado;
- continuidad temporal;
- cantidad de observaciones en el merge final;
- cobertura de activos (`ric`) en la base combinada.

#### Output

Se generan:
- la serie diaria del mercado (`market_prices_daily`);
- la base conjunta equity–mercado (`equity_market_returns_daily`),

que será utilizada para la estimación de betas, shocks agregados y medidas de riesgo sistemático.

In [ ]:
# ======================
# mercado s&p 500 -> serie diaria + merge con retornos de equity
# ======================

# candidatos de mercado (orden de preferencia)
MARKET_CANDIDATES = [".SPXT", ".SPX", "SPY.P", "IVV.P", "VOO.P"]


def fetch_market_series(candidates: list[str]) -> tuple[str, pd.DataFrame]:
    # intenta descargar una serie diaria de mercado y devuelve (ric_elegido, df)
    last_err: Optional[Exception] = None

    for ric in candidates:
        try:
            ts = ek.get_timeseries(
                ric,
                fields=["CLOSE"],
                start_date=START_DATE,
                end_date=END_DATE,
                interval="daily",
            )

            if ts is not None and not ts.empty:
                logger.info(f"Fuente de mercado elegida: {ric}")
                return ric, ts

        except Exception as e:
            last_err = e
            logger.warning(f"Intento fallido con {ric}: {type(e).__name__}: {e}")

        # pausa entre intentos
        sleep_request(REQUEST_PAUSE_SEC)

    raise RuntimeError(
        "No se pudo descargar ninguna serie de mercado. "
        f"Último error: {type(last_err).__name__}: {last_err}" if last_err else
        "No se pudo descargar ninguna serie de mercado."
    )


# -------------------------------------------------
# 1) descarga de serie de mercado
# -------------------------------------------------
chosen_ric, mkt_ts = fetch_market_series(MARKET_CANDIDATES)

# -------------------------------------------------
# 2) construcción de retornos de mercado
# -------------------------------------------------
mkt = (
    mkt_ts
    .rename_axis(index="date")
    .reset_index()
    .rename(columns={"CLOSE": "mkt_close"})
)

mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")

# eliminar timezone si existe
if getattr(mkt["date"].dt, "tz", None) is not None:
    mkt["date"] = mkt["date"].dt.tz_localize(None)

mkt = mkt.sort_values("date").reset_index(drop=True)

# asegurar tipo numérico
mkt["mkt_close"] = pd.to_numeric(mkt["mkt_close"], errors="coerce")

# retorno log de mercado
mkt["mkt_ret_log"] = np.log(mkt["mkt_close"]).diff()


# -------------------------------------------------
# 3) guardado de serie de mercado
# -------------------------------------------------
mkt_parquet = OUT.get("market_prices_daily", RAW_DIR / "market_sp500_daily.parquet")
mkt_csv = mkt_parquet.with_suffix(".csv")

safe_write_parquet(mkt, mkt_parquet, index=False)
mkt.to_csv(mkt_csv, index=False)

logger.info(f"Output -> {mkt_csv.as_posix()}")


# -------------------------------------------------
# 4) merge equity + mercado
# -------------------------------------------------
eq_ret_path = OUT.get("equity_returns_daily", CLEAN_DIR / "equity_returns_daily.parquet")

eq_rets = (
    pd.read_parquet(eq_ret_path)[["date", "ric", "ret_log"]]
    .copy()
)

eq_rets["date"] = pd.to_datetime(eq_rets["date"], errors="coerce")

# eliminar timezone si existe
if getattr(eq_rets["date"].dt, "tz", None) is not None:
    eq_rets["date"] = eq_rets["date"].dt.tz_localize(None)

eq_mkt = (
    eq_rets
    .merge(mkt[["date", "mkt_ret_log"]], on="date", how="inner")
    .dropna(subset=["ret_log", "mkt_ret_log"])
    .sort_values(["ric", "date"])
    .reset_index(drop=True)
)

logger.info(f"Merge equity+mercado | filas={len(eq_mkt):,} | rics={eq_mkt['ric'].nunique():,}")


# -------------------------------------------------
# 5) export final
# -------------------------------------------------
eq_mkt_parquet = OUT.get(
    "equity_market_returns_daily",
    CLEAN_DIR / "equity_market_returns_daily.parquet"
)

safe_write_parquet(eq_mkt, eq_mkt_parquet, index=False)

### Descarga de precios diarios de ETFs sectoriales

En esta sección se descargan series diarias de precios para un conjunto de ETFs sectoriales representativos del mercado estadounidense. Estas series permiten construir referencias sectoriales homogéneas y comparables, útiles para análisis descriptivos, controles sectoriales y eventuales medidas de exposición relativa.

#### Objetivo

Obtener series diarias de precios de cierre para ETFs sectoriales con cobertura suficiente a lo largo del período muestral, preservando trazabilidad y consistencia temporal.

#### Procedimiento

Se define un conjunto de ETFs sectoriales y se consulta Refinitiv para descargar sus precios diarios de cierre (`CLOSE`). La descarga se realiza ETF por ETF e incorpora reintentos automáticos ante errores de rate limit, aplicando pausas y backoff progresivo para reducir problemas de disponibilidad de la API.

Una vez descargadas las series, se normalizan nombres de columnas, formatos de fecha y tipos numéricos, y se identifica cada observación mediante el correspondiente `sector_ric`.

#### Control de calidad

Se registran:
- ETFs con datos válidos;
- ETFs sin datos disponibles;
- rango temporal cubierto por cada serie;
- número de observaciones por ETF.

Estos controles permiten evaluar la calidad y continuidad temporal de cada benchmark sectorial.

#### Output

El resultado se consolida en un único dataset diario y se guarda en formatos parquet y CSV como `sector_etfs_daily`, que podrá utilizarse en etapas posteriores como referencia sectorial de mercado.

In [ ]:
# ======================
# etfs sectoriales -> precios diarios (close)
# ======================

# lista de etfs sectoriales
SECTOR_ETFS = [
    "XLF",     # financials
    "XLE",     # energy
    "XLK",     # technology
    "XLY",     # consumer discretionary
    "XLP",     # consumer staples
    "XLI",     # industrials
    "XLV",     # health care
    "XLU",     # utilities
    "XLRE.K",  # real estate
    "XLB",     # materials
    "XLC",     # communication services
]


def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """
    Descarga una serie diaria con reintentos ante errores de rate limit.
    Devuelve un DataFrame o None si no se pudo obtener información.
    """
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            # si aparece rate limit, aplicar backoff y reintentar
            if "429" in msg or "Too many requests" in msg:
                wait_sec = int(RETRY_BACKOFF_SEC * attempt * 2)
                logger.warning(
                    f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}"
                )
                time.sleep(wait_sec)
                continue

            # otros errores se registran y no se reintentan
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


# -------------------------------------------------
# 1) descarga de series sectoriales
# -------------------------------------------------
sec_frames: list[pd.DataFrame] = []
missing: list[str] = []

logger.info(f"Descarga ETFs sectoriales | total={len(SECTOR_ETFS)}")

for ric in SECTOR_ETFS:
    logger.info(f"ETF sectorial: {ric}")

    ts = get_ts_safe(
        ric,
        START_DATE,
        END_DATE,
        fields=["CLOSE"],
        max_attempts=RETRY_MAX,
    )

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {ric}")
        missing.append(ric)
        sleep_request(0.5)
        continue

    # normalización estándar
    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "price"})
    df["sector_ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    # log resumido de cobertura temporal
    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {ric:<6} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    sec_frames.append(df)

    # pausa breve entre requests
    sleep_request(0.5)

if not sec_frames:
    raise RuntimeError(
        "No se pudo descargar ningún ETF sectorial (todas las series vinieron vacías)."
    )

sectors_ts = pd.concat(sec_frames, ignore_index=True)


# -------------------------------------------------
# 2) guardado de resultados
# -------------------------------------------------
out_parquet = OUT.get("sector_etfs_daily", RAW_DIR / "sector_etfs_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(sectors_ts, out_parquet, index=False)
sectors_ts.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")


# -------------------------------------------------
# 3) resumen de cobertura
# -------------------------------------------------
logger.info(f"ETFs con datos: {sectors_ts['sector_ric'].nunique():,}")
logger.info(f"Filas totales: {len(sectors_ts):,}")

if missing:
    logger.warning(f"ETFs sin datos (muestra): {missing[:10]}")

summary = (
    sectors_ts
    .groupby("sector_ric")["date"]
    .agg(["min", "max", "count"])
    .sort_values("count", ascending=False)
)

logger.info("Cobertura por ETF (top):")
logger.info(summary.head(12).to_string())

### Descarga trimestral de fundamentals desde Refinitiv

En esta sección se descargan variables contables y financieras de frecuencia trimestral para las firmas del universo definido previamente. Estas series constituyen la base para la construcción de controles firm-level, proxies de estructura financiera y variables operativas utilizadas en el panel final.

#### Objetivo

Obtener una base homogénea de fundamentals trimestrales para el período 2015–2025, con cobertura suficiente por firma y una estructura temporal consistente para su posterior integración al panel.

#### Procedimiento

A partir del universo de empresas, se define el identificador de consulta (`ric_query`) que se utilizará para la descarga en Refinitiv. En los casos en que ciertos tickers presentan problemas conocidos de consulta, se aplican overrides puntuales de RIC para mejorar la cobertura efectiva.

La descarga se realiza por batches para mitigar restricciones de la API y se incorporan reintentos automáticos ante errores de rate limit. Una vez obtenidos los datos, se normalizan los nombres de columnas y se reconstruye el vínculo entre cada observación descargada y el ticker original del universo.

Dado que la respuesta de Refinitiv no siempre devuelve una estructura temporal directamente utilizable y homogénea entre firmas, cada serie se alinea posteriormente a una grilla trimestral común para el período 2015–2025. Este paso busca estandarizar la dimensión temporal del panel y facilitar el merge posterior con otras fuentes de datos.

#### Variables descargadas

Entre las variables incluidas se encuentran activos, pasivos, patrimonio, activos y pasivos corrientes, deuda total, caja e inversiones de corto plazo, deuda de corto y largo plazo, ingresos, EBITDA, flujo de caja operativo, gastos por intereses y capex.

#### Control de calidad

Se monitorean:
- RICs con y sin fundamentals disponibles;
- cantidad esperada de trimestres por firma;
- duplicados a nivel (`ric`, `date`);
- cobertura final del universo descargado.

#### Output

El resultado se guarda en formatos parquet y CSV como `fundamentals_extended_q`, y constituye la base de información contable trimestral para la construcción del panel maestro.

In [ ]:
# ======================
# fundamentals -> descarga trimestral (q) por batches
# ======================

import math
import time
from typing import Iterable

# overrides puntuales de ric para consultas (casos conocidos)
RIC_OVERRIDE = {
    "PEP": "PEP.O",
    "ABBV": "ABBV.K",
    "HON": "HON.O",
    "AAPL": "AAPL.O",
    "CSCO": "CSCO.O",
    "ORCL": "ORCL.K",
    "META": "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT": "MSFT.O",
}

# fields de fundamentals extendidos
FUND_FIELDS = [
    "TR.F.TotAssets",
    "TR.F.TotLiab",
    "TR.F.TotShHoldEq",
    "TR.F.TotCurrAssets",
    "TR.F.TotCurrLiab",
    "TR.TotalDebtActValue",
    "TR.F.CashSTInvst",
    "TR.F.STDebtCurrPortOfLTDebt",
    "TR.F.DebtLTTot",
    "TR.F.TotRevenue",
    "TR.F.EBITDA",
    "TR.F.NetCashFlowOp",
    "TR.IntExp",
    "TR.F.CAPEXTot",
]

FUND_PARAMS = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

BATCH_SIZE = 20
SLEEP_BATCH_SEC = 1.0
MAX_RETRY = 3


def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    # divide una lista en batches de tamaño n
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(
    rics_batch: list[str],
    fields: list[str],
    params: dict,
    max_retry: int = 3,
) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante errores de rate limit.
    Devuelve un DataFrame, que puede venir vacío.
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)

            if err is not None and len(err):
                logger.warning(
                    f"Refinitiv devolvió warnings/errores (muestra): {str(err[:2])}"
                )

            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(
                    f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}"
                )
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def to_snake(col: str) -> str:
    # normaliza nombres de columnas a snake_case simple
    x = col.replace("TR.", "").replace("F.", "").replace(":", ".")
    x = x.replace(".", "_")
    return x.lower()


# -------------------------------------------------
# 1) universo y ric de consulta
# -------------------------------------------------
df_emp = (
    pd.read_csv(UNIVERSE_FILE, dtype=str)
    .dropna(subset=["Ticker"])
    .reset_index(drop=True)
)

df_emp["ticker"] = df_emp["Ticker"].astype(str).str.strip()
df_emp["ric_query"] = df_emp["ticker"].map(RIC_OVERRIDE).fillna(df_emp["ticker"])

rics = df_emp["ric_query"].dropna().astype(str).unique().tolist()

logger.info(f"Empresas en universo: {len(df_emp):,}")
logger.info(f"RICs únicos de consulta: {len(rics):,}")


# -------------------------------------------------
# 2) descarga de fundamentals por batches
# -------------------------------------------------
all_frames: list[pd.DataFrame] = []

logger.info("Descargando fundamentals trimestrales por batches...")

for i, batch in enumerate(chunked(rics, BATCH_SIZE), start=1):
    logger.info(f"Batch {i} | size={len(batch)}")

    df_batch = get_data_safe(batch, FUND_FIELDS, FUND_PARAMS, max_retry=MAX_RETRY)

    if not df_batch.empty:
        all_frames.append(df_batch)

    time.sleep(SLEEP_BATCH_SEC)

if not all_frames:
    raise RuntimeError("No se descargaron fundamentals. Revisar fields, permisos o conexión.")

fund = pd.concat(all_frames, ignore_index=True)


# -------------------------------------------------
# 3) normalización de columnas
# -------------------------------------------------
if "Instrument" in fund.columns:
    fund = fund.rename(columns={"Instrument": "ric"})
elif "RIC" in fund.columns:
    fund = fund.rename(columns={"RIC": "ric"})

fund.columns = [to_snake(c) for c in fund.columns]


# -------------------------------------------------
# 4) mapeo de rastreo al ticker original
# -------------------------------------------------
df_map = df_emp[["ticker", "ric_query"]].rename(columns={"ric_query": "ric"})
fund = fund.merge(df_map, on="ric", how="left")


# -------------------------------------------------
# 5) alineación a grilla trimestral común
# -------------------------------------------------
start_q = pd.Timestamp("2015-01-01")
end_q = pd.Timestamp("2025-12-31")
full_q_range = pd.date_range(start=start_q, end=end_q, freq="Q")


def assign_quarter_grid(g: pd.DataFrame) -> pd.DataFrame:
    # asigna una grilla trimestral común a cada firma
    g = g.reset_index(drop=True)
    n = len(g)

    if n == 0:
        return g

    reps = math.ceil(n / len(full_q_range))
    dates = list(full_q_range) * reps
    g["date"] = dates[:n]
    return g


fund = fund.groupby("ric", group_keys=False).apply(assign_quarter_grid)

# eliminar posibles duplicados a nivel firma-trimestre
fund = (
    fund
    .sort_values(["ric", "date"])
    .groupby(["ric", "date"], as_index=False)
    .first()
)


# -------------------------------------------------
# 6) control de calidad
# -------------------------------------------------
logger.info(f"Fundamentals final | filas={len(fund):,} | rics={fund['ric'].nunique():,}")
logger.info(f"Trimestres esperados en grilla: {len(full_q_range):,}")

missing = df_emp.loc[
    ~df_emp["ric_query"].isin(fund["ric"]),
    ["Issuer", "ticker", "ric_query"],
]

if not missing.empty:
    logger.warning(f"Empresas sin fundamentals: {len(missing):,}")
    logger.warning(missing.head(20).to_string(index=False))


# -------------------------------------------------
# 7) export de resultados
# -------------------------------------------------
out_parquet = OUT.get("fundamentals_q", RAW_DIR / "fundamentals_extended_q.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(fund, out_parquet, index=False)
fund.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")

### Descarga diaria de la curva soberana en USD (UST) por tenor

En esta sección se descargan las series diarias de yields de la curva soberana de Estados Unidos para distintos tenores. Estas series constituyen una referencia central para la construcción de benchmarks libres de riesgo y para el análisis de spreads de crédito en bonos corporativos denominados en USD.

#### Objetivo

Obtener una base diaria de tasas soberanas en dólares, con cobertura suficiente a lo largo del período muestral y con desagregación por tenor, de modo de disponer de una curva de referencia consistente para etapas posteriores del análisis.

#### Procedimiento

Se define un conjunto de RICs correspondientes a distintos puntos de la curva U.S. Treasury, desde el tramo corto hasta el tramo largo. Para cada tenor, se descarga la serie diaria de `CLOSE` desde Refinitiv.

La descarga se realiza tenor por tenor e incorpora reintentos automáticos con backoff en caso de errores de rate limit. Una vez obtenidas las series, se normalizan nombres de columnas, formato de fechas y tipo numérico del yield, y se consolida la información en un único dataset.

#### Control de calidad

Se registran:
- tenores con datos válidos;
- tenores sin datos disponibles;
- rango temporal cubierto por cada tenor;
- número de observaciones por serie.

Estas validaciones permiten verificar la consistencia de la curva y detectar eventuales discontinuidades en la cobertura.

#### Output

El resultado se guarda en formatos parquet y CSV como `ust_yield_curve_daily`, y constituye la base soberana de referencia para la medición de spreads y comparaciones de estructura temporal de tasas.

In [ ]:
# ======================
# ust yield curve -> daily (por tenor)
# ======================

UST_RICS = {
    "US1M": "US1MT=RR",
    "US3M": "US3MT=RR",
    "US6M": "US6MT=RR",
    "US1Y": "US1YT=RR",
    "US2Y": "US2YT=RR",
    "US3Y": "US3YT=RR",
    "US5Y": "US5YT=RR",
    "US7Y": "US7YT=RR",
    "US10Y": "US10YT=RR",
    "US30Y": "US30YT=RR",
}


def get_ts_safe(
    ric: str,
    start_date: str,
    end_date: str,
    fields: list[str] | None = None,
    max_attempts: int = 3,
) -> Optional[pd.DataFrame]:
    """
    Descarga una serie diaria con reintentos ante errores de rate limit.
    Devuelve un DataFrame o None si no se pudo obtener información.
    """
    if fields is None:
        fields = ["CLOSE"]

    for attempt in range(1, max_attempts + 1):
        try:
            ts = ek.get_timeseries(
                ric,
                fields=fields,
                start_date=start_date,
                end_date=end_date,
                interval="daily",
            )
            return ts

        except Exception as e:
            msg = str(e)

            # si aparece rate limit, aplicar backoff y reintentar
            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(
                    f"429 para {ric} | espero {wait_sec}s | intento {attempt}/{max_attempts}"
                )
                time.sleep(wait_sec)
                continue

            # otros errores se registran y no se reintentan
            logger.warning(f"Error con {ric}: {type(e).__name__}: {e}")
            return None

    return None


# -------------------------------------------------
# 1) descarga de la curva ust por tenor
# -------------------------------------------------
logger.info("Descargando curva UST (USD) por tenor...")
logger.info(f"Tenores definidos: {list(UST_RICS.keys())}")

ust_frames: list[pd.DataFrame] = []
missing: list[str] = []

for tenor, ric in UST_RICS.items():
    logger.info(f"UST {tenor} | {ric}")

    ts = get_ts_safe(
        ric,
        START_DATE,
        END_DATE,
        fields=["CLOSE"],
        max_attempts=RETRY_MAX,
    )

    if ts is None or ts.empty:
        logger.warning(f"Sin datos: {tenor} ({ric})")
        missing.append(tenor)
        sleep_request(0.4)
        continue

    # normalización estándar
    df = ts.reset_index().rename(columns={"Date": "date", "CLOSE": "yield"})
    df["tenor"] = tenor
    df["ric"] = ric

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if getattr(df["date"].dt, "tz", None) is not None:
        df["date"] = df["date"].dt.tz_localize(None)

    df["yield"] = pd.to_numeric(df["yield"], errors="coerce")

    # log resumido de cobertura por tenor
    dmin = df["date"].min()
    dmax = df["date"].max()
    logger.info(f"OK {tenor:<4} | {dmin.date()} -> {dmax.date()} | filas={len(df):,}")

    ust_frames.append(df)

    # pausa breve entre consultas
    sleep_request(0.4)

if not ust_frames:
    raise RuntimeError("No se pudo descargar ningún punto de la curva UST.")

ust = pd.concat(ust_frames, ignore_index=True)


# -------------------------------------------------
# 2) guardado de resultados
# -------------------------------------------------
out_parquet = OUT.get("ust_yield_curve_daily", RAW_DIR / "ust_yield_curve_daily.parquet")
out_csv = out_parquet.with_suffix(".csv")

safe_write_parquet(ust, out_parquet, index=False)
ust.to_csv(out_csv, index=False)

logger.info(f"Output -> {out_csv.as_posix()}")


# -------------------------------------------------
# 3) resumen de cobertura
# -------------------------------------------------
logger.info(f"Tenores con datos: {ust['tenor'].nunique():,}")
logger.info(f"Filas totales: {len(ust):,}")
logger.info(f"Rango global: {ust['date'].min().date()} -> {ust['date'].max().date()}")

if missing:
    logger.warning(f"Tenores sin datos: {missing}")

### Descarga del histórico mensual de OAS por bono

En esta sección se descarga el histórico mensual de option-adjusted spreads (OAS) para cada bono incluido en el universo previamente construido. Esta base constituye el insumo central para la medición de spreads de crédito a nivel instrumento y para la posterior agregación al nivel firma-tiempo.

#### Objetivo

Obtener series mensuales de OAS para el universo de bonos corporativos elegibles, preservando el identificador específico de cada instrumento (`bond_ric`) y registrando de forma explícita la cobertura efectiva de la descarga.

#### Procedimiento

A partir del universo de bonos filtrado por emisores del S&P 500, se extrae el conjunto de `bond_ric` a consultar en Refinitiv. La descarga se realiza en batches para reducir problemas operativos asociados a límites de consulta y disponibilidad de la API.

Para cada batch, se solicita la serie mensual de `TR.OPTIONADJUSTEDSPREADBID` en el período 2015–2025. El proceso incorpora reintentos automáticos y registra errores a nivel batch e instrumento, permitiendo distinguir entre bonos con datos válidos, bonos fallidos y bonos sin cobertura efectiva.

Una vez descargadas las series, se normalizan las variables de fecha y spread, y se integra nuevamente la información identificatoria del bono y del emisor (`Issuer`, `Ticker`, `bond_ric`).

#### Control de calidad

Se monitorean:
- cantidad total de `bond_ric` consultados;
- batches exitosos y fallidos;
- bonos con y sin OAS descargado;
- rango temporal cubierto por cada bono;
- cantidad de observaciones por instrumento.

Además, se generan reportes auxiliares de cobertura, errores de descarga y logs por batch para facilitar trazabilidad y auditoría.

#### Output

El resultado se guarda como una base mensual a nivel bono en formatos parquet y CSV, junto con archivos auxiliares de cobertura y control de errores. Esta base constituye el insumo principal para la construcción de la variable dependiente de spreads de crédito en el panel empírico.

In [ ]:
# ======================
# oas -> histórico mensual por bono (batch = 25 rics)
# ======================

# validar disponibilidad del universo de bonos en memoria
if "df_bonds_universe" not in globals():
    raise RuntimeError(
        "df_bonds_universe no existe en memoria. "
        "Primero corré la celda que construye el universo de bonos desde "
        "'data/inputs/new_bonds/' y lo filtra por el universo S&P 500."
    )

if "bond_ric" not in df_bonds_universe.columns:
    raise RuntimeError(
        "df_bonds_universe no tiene columna 'bond_ric'. "
        "Revisá la celda de bonos para preservar el RIC del bono."
    )

bonds_full = df_bonds_universe.copy()

bond_rics = (
    bonds_full["bond_ric"]
    .dropna()
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    .dropna()
    .unique()
    .tolist()
)

logger.info(f"Bond RICs para OAS: {len(bond_rics):,}")

if len(bond_rics) == 0:
    raise RuntimeError("La lista de bond_rics quedó vacía.")


# -------------------------------------------------
# helper para dividir en batches
# -------------------------------------------------
def chunk_list(lst, n):
    # divide una lista en bloques de tamaño n
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


BATCH_SIZE = 25
MAX_RETRIES = 3

oas_frames = []
failed_rics = []
batch_logs = []

total_batches = int(np.ceil(len(bond_rics) / BATCH_SIZE))


# -------------------------------------------------
# descarga por batches
# -------------------------------------------------
for bnum, batch in enumerate(chunk_list(bond_rics, BATCH_SIZE), start=1):
    success = False
    batch_start = time.time()
    batch_errs = []

    logger.info(f"OAS batch {bnum}/{total_batches} | tamaño={len(batch)}")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df, err = ek.get_data(
                batch,
                ["TR.OPTIONADJUSTEDSPREADBID.date", "TR.OPTIONADJUSTEDSPREADBID"],
                parameters={
                    "SDate": START_DATE,
                    "EDate": END_DATE,
                    "Frq": "M",
                },
            )

            n_rows = 0 if df is None else len(df)
            n_errs = 0 if err is None else len(err)

            logger.info(
                f"Batch {bnum}/{total_batches} | intento {attempt} | filas={n_rows:,} | errores={n_errs:,}"
            )

            # registrar errores individuales
            if err is not None and len(err):
                for e in err:
                    batch_errs.append(e)

                    row = e.get("row")
                    if row is not None and 0 <= row < len(batch):
                        failed_rics.append(batch[row])

            if df is not None and len(df) > 0:
                oas_frames.append(df)

            success = True
            break

        except Exception as e:
            msg = str(e)
            logger.warning(f"Batch {bnum}/{total_batches} | intento {attempt} falló: {msg}")

            # tratamiento más suave de errores de backend / bad request
            if "400" in msg or "Backend error" in msg:
                time.sleep(10 * attempt)
            else:
                time.sleep(3 * attempt)

    if not success:
        logger.warning(f"Batch fallido: {bnum}/{total_batches}")
        failed_rics.extend(batch)

    batch_time = time.time() - batch_start

    batch_logs.append({
        "batch_num": bnum,
        "batch_size": len(batch),
        "success": success,
        "seconds": round(batch_time, 2),
        "n_errs": len(batch_errs),
    })

    if batch_errs:
        logger.warning(f"Batch {bnum}/{total_batches} | errores registrados: {len(batch_errs):,}")

    # log de progreso
    if bnum % 10 == 0 or bnum == total_batches:
        covered = min(bnum * BATCH_SIZE, len(bond_rics))
        logger.info(
            f"OAS progreso: batch {bnum:,}/{total_batches:,} | bonos aprox. {covered:,}/{len(bond_rics):,}"
        )

    # pausa entre batches
    sleep_request(1.5)


# -------------------------------------------------
# consolidación final
# -------------------------------------------------
oas_all = pd.concat(oas_frames, ignore_index=True) if oas_frames else pd.DataFrame()

logger.info(f"OAS descargado | shape={oas_all.shape}")

if oas_all.empty:
    logger.warning("No se descargó ningún OAS.")

else:
    # normalización de variables
    oas_all["Date"] = pd.to_datetime(oas_all["Date"], errors="coerce")
    if getattr(oas_all["Date"].dt, "tz", None) is not None:
        oas_all["Date"] = oas_all["Date"].dt.tz_localize(None)

    oas_all["Option Adjusted Spread Bid"] = pd.to_numeric(
        oas_all["Option Adjusted Spread Bid"],
        errors="coerce",
    )

    oas_all = oas_all.rename(columns={
        "Instrument": "bond_ric",
        "Date": "date",
        "Option Adjusted Spread Bid": "oas_bps",
    })

    # merge con identificadores del bono y del emisor
    oas_all = oas_all.merge(
        bonds_full[["Issuer", "Ticker", "bond_ric"]].drop_duplicates(),
        on="bond_ric",
        how="left",
        validate="m:1",
    )

    oas_all = oas_all.sort_values(["bond_ric", "date"]).reset_index(drop=True)

    # reporte de cobertura por bono
    coverage = (
        oas_all.groupby("bond_ric")
        .agg(
            first_date=("date", "min"),
            last_date=("date", "max"),
            n_obs=("oas_bps", "count"),
        )
        .reset_index()
    )

    logger.info(f"Bonos con OAS descargado: {oas_all['bond_ric'].nunique():,}")
    logger.info(f"Filas OAS totales: {len(oas_all):,}")

    # -------------------------------------------------
    # export de outputs principales y auxiliares
    # -------------------------------------------------
    safe_write_parquet(
        oas_all,
        OUT["oas_spreads_monthly_bond"],
        index=False,
    )

    oas_all.to_csv(
        OUT["oas_spreads_monthly_bond"].with_suffix(".csv"),
        index=False,
    )

    coverage.to_csv(
        OUT["oas_spreads_monthly_bond"].with_name("oas_coverage_report.csv"),
        index=False,
    )

    pd.DataFrame({"bond_ric": sorted(set(failed_rics))}).to_csv(
        OUT["oas_spreads_monthly_bond"].with_name("oas_failed_rics.csv"),
        index=False,
    )

    pd.DataFrame(batch_logs).to_csv(
        OUT["oas_spreads_monthly_bond"].with_name("oas_batch_logs.csv"),
        index=False,
    )

    logger.info(f"Guardado OAS en: {OUT['oas_spreads_monthly_bond']}")
    logger.info(
        f"Bond RICs con OAS: {oas_all['bond_ric'].nunique():,} / {len(bond_rics):,}"
    )
    logger.info(f"Bond RICs sin OAS o fallidos: {len(set(failed_rics)):,}")

### Enriquecimiento del panel mensual de bonos con fecha de vencimiento y madurez residual

En esta sección se integra al panel mensual de bonos la metadata estructural del instrumento, en particular su fecha de vencimiento contractual (`Maturity`) y la medida derivada de madurez residual (`residual_maturity_years`).

#### Objetivo

Preservar, junto con la serie mensual de OAS, la información básica del bono necesaria para caracterizar su estructura temporal y controlar por heterogeneidad entre instrumentos dentro del panel.

#### Procedimiento

Se parte de dos fuentes complementarias. Por un lado, el universo de bonos construido previamente a partir de exportaciones amplias de Workspace contiene la metadata del instrumento, incluyendo su fecha de vencimiento. Por otro, la base mensual de OAS descargada desde Refinitiv aporta la dimensión temporal observada a nivel bono-mes.

Primero se normalizan identificadores y nombres de columnas en ambas fuentes, y se seleccionan las variables relevantes de metadata. Luego, la base de OAS se consolida a nivel (`bond_ric`, `date`) para evitar duplicaciones. Finalmente, ambas fuentes se integran mediante `bond_ric`, preservando el identificador del bono y la información del emisor.

A partir de la fecha de observación mensual y de la fecha de vencimiento contractual, se calcula la madurez residual como:

$$
\text{residual\_maturity\_years}_{i,t} =
\frac{\text{Maturity}_i - \text{date}_t}{365.25}
$$

Cuando la diferencia resulta negativa, la observación se trata como faltante, dado que corresponde a bonos cuya fecha de vencimiento ya fue superada.

#### Control de calidad

Se verifican:
- cobertura de `Maturity` en la metadata de bonos;
- cobertura de OAS en la base mensual;
- cantidad de bonos únicos en ambas fuentes;
- duplicados potenciales a nivel (`bond_ric`, `date`);
- cobertura final de `residual_maturity_years`.

#### Output

El resultado se guarda como `bonds_monthly_spreads` en formatos parquet y CSV. Aunque se preserva el nombre histórico del archivo por compatibilidad con el pipeline, en esta versión el contenido corresponde a un panel mensual por bono que combina OAS, metadata del instrumento y madurez residual derivada.

In [ ]:
# ==========================================================
# enriquecer bonos con maturity y residual maturity
# ==========================================================

import pandas as pd
import numpy as np

# ----------------------------------------------------------
# paths de salida
# ----------------------------------------------------------
OUT_PARQUET = CLEAN_DIR / "bonds_monthly_spreads.parquet"
OUT_CSV = CLEAN_DIR / "bonds_monthly_spreads.csv"


# ----------------------------------------------------------
# helpers
# ----------------------------------------------------------
def to_eom(s):
    # lleva fechas al fin de mes para homogeneizar frecuencia mensual
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp("M")


def pick_col(df, candidates):
    # selecciona la primera columna disponible entre varios candidatos
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        if str(cand).lower() in lower_map:
            return lower_map[str(cand).lower()]

    return None


def clean_ric(s):
    # limpieza estándar de identificadores RIC
    s = s.astype(str).str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NAN": pd.NA,
        "NONE": pd.NA,
    })
    return s


# ----------------------------------------------------------
# 1) validaciones básicas
# ----------------------------------------------------------
if "df_bonds_universe" not in globals():
    raise ValueError(
        "No encontré 'df_bonds_universe' en memoria. "
        "Primero corré la celda que construye el universo de bonos desde "
        "'data/inputs/new_bonds/' y lo filtra por el universo S&P 500."
    )

oas_path = (
    OUT["oas_spreads_monthly_bond"]
    if "oas_spreads_monthly_bond" in OUT
    else (RAW_DIR / "oas_spreads_monthly_bond.parquet")
)

if not oas_path.exists():
    raise FileNotFoundError(f"No encontré el archivo de OAS: {oas_path}")

oas = pd.read_parquet(oas_path).copy()
bonds_meta = df_bonds_universe.copy()


# ----------------------------------------------------------
# 2) normalizar metadata de bonos
# ----------------------------------------------------------
ric_col = pick_col(bonds_meta, ["bond_ric", "Preferred RIC", "RIC", "Instrument", "Issue RIC"])
if ric_col is None:
    raise ValueError(
        f"No encontré columna de RIC en df_bonds_universe. Columnas: {list(bonds_meta.columns)}"
    )

mat_col = pick_col(bonds_meta, ["Maturity", "Maturity Date", "maturity"])
if mat_col is None:
    raise ValueError(
        f"No encontré columna de Maturity en df_bonds_universe. Columnas: {list(bonds_meta.columns)}"
    )

issuer_col = pick_col(bonds_meta, ["Issuer"])
ticker_col = pick_col(bonds_meta, ["Ticker"])
isin_col = pick_col(bonds_meta, ["ISIN"])
coupon_col = pick_col(bonds_meta, ["Coupon"])
curr_col = pick_col(bonds_meta, ["Principal_Currency", "Principal Currency"])
amt_col = pick_col(bonds_meta, ["Amount_Issued_USD", "Amount Issued (USD)"])
src_col = pick_col(bonds_meta, ["source_file"])

rename_map = {
    ric_col: "bond_ric",
    mat_col: "Maturity",
}

if issuer_col is not None:
    rename_map[issuer_col] = "Issuer"
if ticker_col is not None:
    rename_map[ticker_col] = "Ticker"
if isin_col is not None:
    rename_map[isin_col] = "ISIN"
if coupon_col is not None:
    rename_map[coupon_col] = "Coupon"
if curr_col is not None:
    rename_map[curr_col] = "Principal_Currency"
if amt_col is not None:
    rename_map[amt_col] = "Amount_Issued_USD"
if src_col is not None:
    rename_map[src_col] = "source_file"

keep_meta = list(rename_map.keys())

bonds_meta = bonds_meta[keep_meta].copy().rename(columns=rename_map)

bonds_meta["bond_ric"] = clean_ric(bonds_meta["bond_ric"])
bonds_meta["Maturity"] = pd.to_datetime(bonds_meta["Maturity"], errors="coerce")

if "Coupon" in bonds_meta.columns:
    bonds_meta["Coupon"] = pd.to_numeric(bonds_meta["Coupon"], errors="coerce")

if "Amount_Issued_USD" in bonds_meta.columns:
    bonds_meta["Amount_Issued_USD"] = pd.to_numeric(
        bonds_meta["Amount_Issued_USD"],
        errors="coerce",
    )

bonds_meta = (
    bonds_meta
    .dropna(subset=["bond_ric"])
    .drop_duplicates(subset=["bond_ric"])
    .reset_index(drop=True)
)

logger.info(f"bonds_meta | filas={len(bonds_meta):,} | rics={bonds_meta['bond_ric'].nunique():,}")
logger.info(f"bonds_meta | proporción sin Maturity={bonds_meta['Maturity'].isna().mean():.4f}")


# ----------------------------------------------------------
# 3) preparar OAS mensual por bono
# ----------------------------------------------------------
oas.columns = [str(c).strip() for c in oas.columns]

ric_o = pick_col(oas, ["bond_ric", "RIC", "Instrument", "Preferred RIC", "Issue RIC"])
date_o = pick_col(oas, ["date", "Date"])
oas_o = pick_col(oas, ["oas", "OAS", "oas_bps", "OAS(bp)", "Option Adjusted Spread Bid", "option_adjusted_spread"])

if ric_o is None or date_o is None:
    raise ValueError(
        f"No encontré columnas mínimas en oas_spreads_monthly_bond. Columnas: {list(oas.columns)}"
    )

oas = oas.rename(columns={
    ric_o: "bond_ric",
    date_o: "date",
})

if oas_o is not None and oas_o != "oas":
    oas = oas.rename(columns={oas_o: "oas"})

oas["bond_ric"] = clean_ric(oas["bond_ric"])
oas["date"] = to_eom(oas["date"])

if "oas" in oas.columns:
    oas["oas"] = pd.to_numeric(oas["oas"], errors="coerce")
else:
    oas["oas"] = pd.NA

# colapsar duplicados potenciales a nivel bono-mes
agg_dict = {}
for c in oas.columns:
    if c in ["bond_ric", "date"]:
        continue
    if c == "oas":
        agg_dict[c] = "mean"
    else:
        agg_dict[c] = "first"

oas_m = oas.groupby(["bond_ric", "date"], as_index=False).agg(agg_dict)

logger.info(f"oas_m | filas={len(oas_m):,} | rics={oas_m['bond_ric'].nunique():,}")
logger.info(f"oas_m | proporción sin oas={oas_m['oas'].isna().mean():.4f}")


# ----------------------------------------------------------
# 4) merge final + residual maturity
# ----------------------------------------------------------
bonds_monthly_spreads = oas_m.merge(
    bonds_meta,
    on="bond_ric",
    how="left",
    validate="m:1",
)

bonds_monthly_spreads["residual_maturity_years"] = (
    (bonds_monthly_spreads["Maturity"] - bonds_monthly_spreads["date"]).dt.days / 365.25
)

# observaciones posteriores al vencimiento se consideran faltantes
bonds_monthly_spreads.loc[
    bonds_monthly_spreads["residual_maturity_years"] < 0,
    "residual_maturity_years",
] = np.nan

logger.info(
    f"bonds_monthly_spreads | filas={len(bonds_monthly_spreads):,} | "
    f"rics={bonds_monthly_spreads['bond_ric'].nunique():,}"
)
logger.info(
    f"bonds_monthly_spreads | proporción sin Maturity="
    f"{bonds_monthly_spreads['Maturity'].isna().mean():.4f}"
)
logger.info(
    f"bonds_monthly_spreads | proporción sin residual_maturity_years="
    f"{bonds_monthly_spreads['residual_maturity_years'].isna().mean():.4f}"
)
logger.info(
    f"bonds_monthly_spreads | proporción sin oas="
    f"{bonds_monthly_spreads['oas'].isna().mean():.4f}"
)


# ----------------------------------------------------------
# 5) guardar resultados
# ----------------------------------------------------------
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

bonds_monthly_spreads.to_parquet(OUT_PARQUET, index=False)
bonds_monthly_spreads.to_csv(OUT_CSV, index=False)

logger.info(f"Output parquet -> {OUT_PARQUET.as_posix()}")
logger.info(f"Output csv -> {OUT_CSV.as_posix()}")

### Descarga de spreads diarios de CDS corporativos

En esta sección se construye la base diaria de spreads de CDS corporativos para las firmas del universo. La descarga se realiza en dos etapas: primero se identifica, para cada emisor, el `Primary CDS RIC` reportado por Refinitiv; luego se consulta el histórico diario de spreads asociado a ese contrato.

#### Objetivo

Obtener una serie diaria de spreads de CDS para los emisores del universo, preservando la trazabilidad entre el identificador del emisor y el identificador específico del contrato CDS consultado.

#### Procedimiento

A partir del universo de firmas, se define el identificador de consulta de cada emisor (`ric`). Cuando el RIC no está disponible directamente, se construye una aproximación a partir del ticker mediante sufijos de mercado estándar.

En una primera etapa, se consulta `TR.CDSPrimaryCDSRic` para identificar el contrato CDS principal asociado a cada emisor. En una segunda etapa, se descarga para cada `primary_cds_ric` el histórico diario del spread (`Par Mid Spread`) en USD para el período de análisis.

Finalmente, la base resultante se integra con la información identificatoria del emisor (`Issuer`, `Ticker`, `ric`) y se ordena cronológicamente.

#### Control de calidad

Se monitorean:
- emisores para los cuales se identifica un `Primary CDS RIC`;
- emisores sin identificador CDS válido;
- cobertura histórica efectiva del spread;
- cantidad total de filas descargadas;
- cantidad de emisores con CDS disponible.

Además, se exporta un archivo auxiliar con los casos de `Primary CDS RIC` inválidos o vacíos.

#### Output

El resultado se guarda como `cds_spreads_daily` en formato parquet y constituye la base diaria de spreads de CDS corporativos para análisis descriptivos, contrastes de robustez y posibles variables dependientes alternativas.

# ======================
# cds -> spreads diarios (primary cds ric + par mid spread)
# ======================

def clean_str_series(s: pd.Series) -> pd.Series:
    # limpieza estándar de strings
    return (
        s.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NAN": pd.NA, "NONE": pd.NA})
    )


def get_primary_cds_ric_batch(rics_batch):
    """
    Devuelve el primary CDS RIC por emisor.
    """
    # limpiar batch antes de enviarlo
    rics_batch = [
        str(x).strip()
        for x in rics_batch
        if pd.notna(x) and str(x).strip() != ""
    ]

    if not rics_batch:
        return pd.DataFrame()

    try:
        df, err = ek.get_data(rics_batch, ["TR.CDSPrimaryCDSRic"])

        if err is not None and len(err):
            logger.warning(f"Warnings en primary CDS batch (muestra): {str(err[:2])}")

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.rename(columns={
            "Instrument": "ric",
            "Primary CDS RIC": "primary_cds_ric",
        }).copy()

        df["ric"] = clean_str_series(df["ric"])
        df["primary_cds_ric"] = clean_str_series(df["primary_cds_ric"])

        return df

    except Exception as e:
        logger.warning(f"Error en primary CDS batch: {type(e).__name__}: {e}")
        return pd.DataFrame()


def get_cds_history_one(cds_ric, start_date, end_date):
    """
    Descarga histórico diario de Par Mid Spread para un CDS RIC.
    """
    cds_ric = "" if pd.isna(cds_ric) else str(cds_ric).strip()

    if cds_ric == "" or cds_ric.upper() in {"NONE", "NAN"}:
        return pd.DataFrame()

    try:
        df, err = ek.get_data(
            [cds_ric],
            ["TR.CDSType", "TR.PARMIDSPREAD.date", "TR.PARMIDSPREAD"],
            {
                "SDate": start_date,
                "EDate": end_date,
                "DateType": "AD",
                "CURN": "USD",
            },
        )

        if err is not None and len(err):
            logger.warning(f"Warnings en histórico CDS {cds_ric} (muestra): {str(err[:2])}")

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.rename(columns={
            "Instrument": "cds_ric",
            "Date": "date",
            "Par Mid Spread": "cds_spread_bps",
            "CDS Type": "cds_type",
        }).copy()

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["cds_spread_bps"] = pd.to_numeric(df["cds_spread_bps"], errors="coerce")
        df["cds_ric"] = clean_str_series(df["cds_ric"])

        return df

    except Exception as e:
        logger.warning(f"Error CDS {cds_ric}: {type(e).__name__}: {e}")
        return pd.DataFrame()


# -------------------------------------------------
# helper local
# -------------------------------------------------
def ticker_to_ric(ticker: str):
    # construcción simple de RIC a partir de ticker
    ticker = str(ticker).strip().upper()

    nasdaq = {"AAPL", "MSFT", "GOOGL", "META", "CSCO", "PEP", "HON"}
    nyse = {
        "JPM", "BAC", "C", "WFC", "GS", "MS", "XOM", "CVX", "COP", "OXY",
        "JNJ", "PFE", "MRK", "ABBV", "PG", "KO", "MCD", "WMT",
        "GE", "MMM", "CAT", "DUK", "NEE", "SO", "ORCL",
    }

    if ticker in nasdaq:
        return f"{ticker}.O"

    if ticker in nyse:
        return f"{ticker}.N"

    return f"{ticker}.N"


# -------------------------------------------------
# 1) universo de emisores
# -------------------------------------------------
meta_path = RAW_DIR / "company_metadata.parquet"

df_univ = pd.read_csv(UNIVERSE_FILE, dtype=str).copy()
df_univ = df_univ.dropna(subset=["Ticker"]).reset_index(drop=True)

if "Issuer" not in df_univ.columns:
    df_univ["Issuer"] = pd.NA

df_univ["Ticker"] = df_univ["Ticker"].astype(str).str.strip()

if "RIC" in df_univ.columns:
    df_univ["RIC"] = clean_str_series(df_univ["RIC"])
else:
    df_univ["RIC"] = pd.NA

df_univ["ric"] = df_univ["RIC"]

mask_missing_ric = df_univ["ric"].isna()
df_univ.loc[mask_missing_ric, "ric"] = (
    df_univ.loc[mask_missing_ric, "Ticker"]
    .astype(str)
    .str.strip()
    .apply(ticker_to_ric)
)

df_univ["ric"] = clean_str_series(df_univ["ric"])

if meta_path.exists():
    logger.info(f"Usando metadata de empresas: {meta_path}")

    meta_emp = pd.read_parquet(meta_path).copy()

    # normalizar nombre de columna del instrumento
    if "Instrument" in meta_emp.columns:
        meta_emp = meta_emp.rename(columns={"Instrument": "ric"})
    elif "RIC" in meta_emp.columns:
        meta_emp = meta_emp.rename(columns={"RIC": "ric"})

    if "ric" not in meta_emp.columns:
        raise RuntimeError("company_metadata.parquet no tiene columna 'ric'.")

    meta_emp["ric"] = clean_str_series(meta_emp["ric"])

    # reconstruir ticker / issuer desde el universo original
    df_emp = (
        meta_emp[["ric"]]
        .drop_duplicates()
        .merge(
            df_univ[["Ticker", "Issuer", "ric"]].drop_duplicates(),
            on="ric",
            how="left",
        )
    )
else:
    logger.warning("No existe company_metadata.parquet. Se usa fallback con Ticker.")
    df_emp = df_univ[["Ticker", "Issuer", "ric"]].drop_duplicates().copy()

df_emp["ric"] = clean_str_series(df_emp["ric"])
df_emp = (
    df_emp
    .dropna(subset=["ric"])
    .drop_duplicates(subset=["ric"])
    .reset_index(drop=True)
)

rics = df_emp["ric"].tolist()

logger.info(f"Emisores a pedir CDS: {len(rics):,}")


# -------------------------------------------------
# 2) primary CDS RIC
# -------------------------------------------------
primary_frames = []

for batch in chunked(rics, 25):
    df_batch = get_primary_cds_ric_batch(batch)
    if not df_batch.empty:
        primary_frames.append(df_batch)

primary_cds = (
    pd.concat(primary_frames, ignore_index=True)
    if primary_frames else pd.DataFrame()
)

if primary_cds.empty:
    raise RuntimeError("No se pudo obtener ningún Primary CDS RIC.")

primary_cds["ric"] = clean_str_series(primary_cds["ric"])
primary_cds["primary_cds_ric"] = clean_str_series(primary_cds["primary_cds_ric"])

primary_cds = (
    primary_cds
    .dropna(subset=["ric", "primary_cds_ric"])
    .drop_duplicates(subset=["ric", "primary_cds_ric"])
    .reset_index(drop=True)
)

logger.info(f"Primary CDS RICs encontrados: {len(primary_cds):,}")


# -------------------------------------------------
# 3) histórico CDS
# -------------------------------------------------
frames = []
invalid_primary = []

for _, row in primary_cds.iterrows():
    cds_ric = "" if pd.isna(row["primary_cds_ric"]) else str(row["primary_cds_ric"]).strip()
    issuer_ric = "" if pd.isna(row["ric"]) else str(row["ric"]).strip()

    if cds_ric == "" or cds_ric.upper() in {"NONE", "NAN"}:
        invalid_primary.append({"ric": issuer_ric, "primary_cds_ric": cds_ric})
        logger.warning(f"Primary CDS vacío o inválido: {issuer_ric} -> {cds_ric}")
        continue

    df = get_cds_history_one(cds_ric, START_DATE, END_DATE)

    if df is not None and not df.empty:
        df["ric"] = issuer_ric
        frames.append(df)
        logger.info(f"OK CDS {issuer_ric} -> {cds_ric} | filas={len(df):,}")
    else:
        logger.warning(f"Sin CDS histórico: {issuer_ric} -> {cds_ric}")

if not frames:
    raise RuntimeError("Ningún emisor devolvió histórico de CDS.")

cds_all = pd.concat(frames, ignore_index=True)

cds_all = cds_all.merge(
    df_emp[["ric", "Issuer", "Ticker"]].drop_duplicates(),
    on="ric",
    how="left",
)

cds_all = cds_all.sort_values(["ric", "date"]).reset_index(drop=True)


# -------------------------------------------------
# 4) guardar resultados
# -------------------------------------------------
safe_write_parquet(cds_all, OUT["cds_spreads_daily"], index=False)
cds_all.to_csv(OUT["cds_spreads_daily"].with_suffix(".csv"), index=False)

if invalid_primary:
    pd.DataFrame(invalid_primary).to_csv(
        OUT["cds_spreads_daily"].with_name("cds_invalid_primary_rics.csv"),
        index=False,
    )

logger.info(f"Filas CDS: {len(cds_all):,}")
logger.info(f"Emisores con CDS: {cds_all['ric'].nunique():,}")
logger.info(f"Guardado en: {OUT['cds_spreads_daily']}")

### Revenue y market share dentro del universo S&P 500

En esta sección se construye una medida trimestral de participación de mercado a partir de ventas reportadas por firmas del S&P 500 y su agregación por industria. Esta variable busca capturar una dimensión observable de posicionamiento relativo dentro del mercado de producto y constituye un insumo relevante para el análisis de heterogeneidad entre firmas.

#### Objetivo

Obtener una medida de `market_share` a nivel firma-tiempo basada en ventas trimestrales, utilizando como referencia el total de ventas del grupo industrial correspondiente dentro del universo S&P 500.

#### Procedimiento

Primero se descarga el universo de constituyentes del S&P 500 desde Refinitiv y se obtiene para cada firma la metadata de clasificación GICS. Luego se descargan fundamentals trimestrales de revenue para ese universo, seleccionando automáticamente el campo de ventas disponible entre un conjunto de candidatos, según cobertura y permisos.

A continuación, cada observación de revenue se empareja con su clasificación GICS y se calcula, para cada trimestre, el total de ventas del grupo industrial (`GICS Industry Group`). La participación de mercado se define entonces como el cociente entre las ventas de la firma y las ventas agregadas de su grupo industrial en ese trimestre.

Posteriormente, la base trimestral se depura para eliminar duplicados a nivel (`ric`, `period_end`) y se transforma a frecuencia mensual mediante forward fill, de modo de integrarse de forma consistente al panel maestro mensual.

#### Definición

La variable de participación de mercado se construye como:

$$
\text{market\_share}_{i,t} =
\frac{\text{revenue}_{i,t}}{\sum_{j \in g(i)} \text{revenue}_{j,t}}
$$

donde \( g(i) \) representa el grupo industrial GICS al que pertenece la firma \( i \).

#### Control de calidad

Se monitorean:
- campo de revenue efectivamente utilizado;
- cobertura de clasificación GICS;
- número de observaciones por trimestre;
- consistencia de `market_share` en el intervalo \([0,1]\) cuando existe información de industria;
- duplicados a nivel firma-trimestre antes de la expansión mensual.

#### Output

Se generan tres productos:

- ventas agregadas por industria y trimestre para el universo S&P 500;
- panel completo del S&P 500 con revenue y market share;
- panel filtrado a la muestra de estudio.

El resultado final utilizado en el pipeline queda expresado en frecuencia mensual para facilitar su integración con el resto de las variables del panel.

In [ ]:
# ======================
# revenue + market share (sp500)
# ======================

# -------------------------------------------------
# helpers de descarga
# -------------------------------------------------
def chunked(items: list[str], n: int) -> Iterable[list[str]]:
    # divide una lista en batches
    for i in range(0, len(items), n):
        yield items[i:i + n]


def get_data_safe(
    rics_batch: list[str],
    fields: list[str],
    params: dict,
    max_retry: int = 3,
) -> pd.DataFrame:
    """
    Wrapper para ek.get_data con reintentos ante errores de rate limit.
    Devuelve un DataFrame, que puede venir vacío.
    """
    for attempt in range(1, max_retry + 1):
        try:
            df, err = ek.get_data(rics_batch, fields=fields, parameters=params)

            if err is not None and len(err):
                logger.warning(f"Refinitiv warnings/errores (muestra): {str(err[:2])}")

            return df if df is not None else pd.DataFrame()

        except Exception as e:
            msg = str(e)

            if "429" in msg or "Too many requests" in msg:
                wait_sec = 4 + (attempt - 1) * 4
                logger.warning(
                    f"429 en batch | espero {wait_sec}s | intento {attempt}/{max_retry}"
                )
                time.sleep(wait_sec)
                continue

            logger.warning(f"Error en batch (no 429): {type(e).__name__}: {e}")
            return pd.DataFrame()

    return pd.DataFrame()


def get_metadata_batched(
    instruments: list[str],
    fields: list[str],
    batch_size: int = 25,
    pause_sec: float = 0.4,
) -> pd.DataFrame:
    """
    Descarga metadata por batches usando ek.get_data sin parameters.
    """
    frames: list[pd.DataFrame] = []

    for i, batch in enumerate(chunked(instruments, batch_size), start=1):
        try:
            df_batch, err = ek.get_data(batch, fields=fields)

            if err is not None and len(err):
                logger.warning(f"Warnings/errores metadata batch (muestra): {str(err[:2])}")

            if df_batch is not None and not df_batch.empty:
                frames.append(df_batch)

        except Exception as e:
            logger.warning(f"Error metadata batch {i} | {type(e).__name__}: {e}")

        time.sleep(pause_sec)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def first_present(cands: list[str], cols_set: set[str]) -> Optional[str]:
    # devuelve el primer candidato presente en el set de columnas
    for c in cands:
        if c in cols_set:
            return c
    return None


# -------------------------------------------------
# 1) muestra de estudio para filtrar al final
# -------------------------------------------------
df_emp = pd.read_csv(UNIVERSE_FILE, dtype=str)
df_emp["Ticker"] = df_emp["Ticker"].astype(str).str.strip()

RIC_FIX = {
    "PEP": "PEP.O",
    "ABBV": "ABBV.K",
    "HON": "HON.O",
    "AAPL": "AAPL.O",
    "CSCO": "CSCO.O",
    "ORCL": "ORCL.K",
    "META": "META.O",
    "GOOGL": "GOOGL.O",
    "MSFT": "MSFT.O",
}


def fix_ticker(t: str) -> str:
    # corrige casos puntuales de ticker/ric para el filtro final
    t = str(t).strip()
    if not t:
        return t
    base = t.split(".")[0]
    return RIC_FIX.get(base, t)


df_emp["Ticker_fixed"] = df_emp["Ticker"].apply(fix_ticker)
df_emp["ric_base"] = df_emp["Ticker_fixed"].str.split(".").str[0]
target_bases = sorted(df_emp["ric_base"].dropna().unique().tolist())

logger.info(f"Muestra de estudio (bases RIC): {len(target_bases):,}")


# -------------------------------------------------
# 2) universo sp500 desde refinitiv
# -------------------------------------------------
logger.info("Descargando universo SP500 (0#.SPX)...")

spx_df, spx_err = ek.get_data("0#.SPX", ["TR.RIC"])

if spx_err is not None and len(spx_err):
    logger.warning(f"Warnings universo SPX (muestra): {str(spx_err[:3])}")

if spx_df is None or spx_df.empty:
    raise RuntimeError("No llegó universo SPX desde Refinitiv.")

spx_df = spx_df.dropna(subset=["RIC"]).copy()
spx_df["RIC"] = spx_df["RIC"].astype(str).str.strip()
spx_rics = spx_df["RIC"].dropna().unique().tolist()

logger.info(f"RICs SP500 detectados: {len(spx_rics):,}")


# -------------------------------------------------
# 3) metadata GICS del universo sp500
# -------------------------------------------------
META_FIELDS = [
    "TR.CommonName",
    "TR.PrimaryRIC",
    "TR.ISIN",
    "TR.CUSIP",
    "TR.PermID",
    "TR.CompanyMarketCap",
    "TR.GICSSector",
    "TR.GICSSectorName",
    "TR.GICSIndustryGroup",
    "TR.GICSIndustryGroupName",
    "TR.GICSIndustry",
    "TR.GICSIndustryName",
    "TR.GICSSubIndustry",
    "TR.GICSSubIndustryName",
    "TR.HeadquartersCountry",
]

logger.info("Descargando metadata GICS para SP500...")

meta_spx = get_metadata_batched(spx_rics, META_FIELDS, batch_size=25, pause_sec=0.4)

if meta_spx.empty:
    raise RuntimeError("No llegó metadata SPX (GICS).")

meta_spx.columns = [str(c).strip() for c in meta_spx.columns]
meta_spx = meta_spx.rename(columns={
    "Instrument": "ric",
    "GICS Sector Name": "gics_sector_name",
    "GICS Industry Group Name": "gics_industry_group_name",
    "GICS Industry Name": "gics_industry_name",
    "GICS Sub-Industry Name": "gics_subindustry_name",
})

meta_spx["ric"] = meta_spx["ric"].astype(str).str.strip()

gics_cols = [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

meta_spx_gics = meta_spx[["ric"] + gics_cols].drop_duplicates(subset=["ric"]).copy()

logger.info(f"Metadata GICS SP500 | filas={len(meta_spx_gics):,}")


# -------------------------------------------------
# 4) fundamentals trimestrales de revenue para sp500
# -------------------------------------------------
REV_CANDS = [
    "TR.F.TotRevenue",
    "TR.F.SalesTurnover",
    "TR.F.TotalRevenue",
    "TR.F.SalesOrRevenue",
    "TR.F.NetSales",
    "TR.F.Revenue",
    "TR.F.RevenueReported",
]

fields_funda = REV_CANDS + ["TR.F.PeriodEndDate"]

params = {
    "SDate": "2015-01-01",
    "EDate": "2025-12-31",
    "Frq": "Q",
    "Curn": "USD",
}

logger.info("Descargando fundamentals de revenue para SP500...")

funda_frames: list[pd.DataFrame] = []
BATCH = 20
SLEEP_SEC = 1.0

for i, batch in enumerate(chunked(spx_rics, BATCH), start=1):
    logger.info(f"Batch fundamentals {i} | size={len(batch)}")

    df_b = get_data_safe(batch, fields_funda, params, max_retry=RETRY_MAX)

    if df_b is not None and not df_b.empty:
        funda_frames.append(df_b)

    time.sleep(SLEEP_SEC)

if not funda_frames:
    raise RuntimeError("No llegó fundamentals de revenue.")

funda_all = pd.concat(funda_frames, ignore_index=True)
funda_all.columns = [str(c).strip() for c in funda_all.columns]
cols = set(funda_all.columns)

ric_col = first_present(["Instrument", "RIC", "TR.RIC", "Constituent RIC"], cols)
ped_col = first_present(["TR.F.PeriodEndDate", "Period End Date", "PeriodEndDate"], cols)
rev_col = first_present(
    REV_CANDS + ["Revenue from Business Activities - Total", "Revenue", "Sales/Turnover"],
    cols,
)

missing = [
    x for x, y in [("ric", ric_col), ("period_end", ped_col), ("revenue", rev_col)]
    if y is None
]
if missing:
    raise RuntimeError(
        f"Columnas faltantes en fundamentals: {missing} | disponibles={sorted(cols)}"
    )

logger.info(f"Campo revenue usado: {rev_col}")

funda_all = funda_all.rename(columns={
    ric_col: "ric",
    ped_col: "period_end",
    rev_col: "revenue",
})

funda_all["ric"] = funda_all["ric"].astype(str).str.strip()
funda_all["period_end"] = pd.to_datetime(funda_all["period_end"], errors="coerce")
funda_all["revenue"] = pd.to_numeric(funda_all["revenue"], errors="coerce")

funda_all = funda_all.dropna(subset=["period_end"]).copy()

logger.info(f"Fundamentals revenue | filas={len(funda_all):,}")


# -------------------------------------------------
# 5) merge con GICS + ventas por industria + market share
# -------------------------------------------------
funda_all = funda_all.merge(meta_spx_gics, on="ric", how="left")

group_col = "gics_industry_group_name"
if group_col not in funda_all.columns:
    raise RuntimeError(
        f"No existe '{group_col}' en funda_all. Columnas={funda_all.columns.tolist()}"
    )

sector_sales = (
    funda_all
    .dropna(subset=[group_col])
    .groupby([group_col, "period_end"], as_index=False)["revenue"]
    .sum()
    .rename(columns={"revenue": "industry_sales"})
)

out_sales = OUT.get(
    "gics_industry_group_sales_q_sp500",
    RAW_DIR / "gics_industry_group_sales_quarterly_sp500.csv",
)
sector_sales.to_csv(out_sales, index=False)

logger.info(f"Output -> {out_sales.as_posix()} | filas={len(sector_sales):,}")

funda_all = funda_all.merge(
    sector_sales,
    on=[group_col, "period_end"],
    how="left",
)

funda_all["market_share"] = funda_all["revenue"] / funda_all["industry_sales"]


# -------------------------------------------------
# 6) export intermedio: sp500 completo + muestra
# -------------------------------------------------
funda_all["ric_base"] = funda_all["ric"].astype(str).str.split(".").str[0]

out_sp500 = OUT.get(
    "fundamentals_with_market_share_sp500",
    RAW_DIR / "fundamentals_with_market_share_sp500.csv",
)
funda_all.to_csv(out_sp500, index=False)

logger.info(f"Output -> {out_sp500.as_posix()} | filas={len(funda_all):,}")

funda_focus = funda_all[funda_all["ric_base"].isin(target_bases)].copy()

out_focus = OUT.get(
    "fundamentals_with_market_share_focus",
    RAW_DIR / "fundamentals_with_market_share.csv",
)
funda_focus.to_csv(out_focus, index=False)

logger.info(f"Output -> {out_focus.as_posix()} | filas={len(funda_focus):,}")

# control básico: market share fuera de rango
bad_ms = funda_all.loc[funda_all["industry_sales"].notna(), "market_share"]
share_oob = ((bad_ms < 0) | (bad_ms > 1)).sum()

if share_oob:
    logger.warning(f"Market share fuera de [0,1] (conteo): {int(share_oob)}")


# ==========================================================
# 7) fix + convertir quarterly -> monthly (forward fill)
# ==========================================================

# colapsar duplicados por ric-period_end
funda_all = (
    funda_all
    .sort_values(["ric", "period_end"])
    .groupby(["ric", "period_end"], as_index=False)
    .agg({
        "revenue": "mean",
        "industry_sales": "mean",
        "market_share": "mean",
        "gics_sector_name": "first",
        "gics_industry_group_name": "first",
        "gics_industry_name": "first",
        "gics_subindustry_name": "first",
    })
)

logger.info(
    f"QA post-collapse | duplicados ric-period_end="
    f"{funda_all.duplicated(['ric', 'period_end']).sum():,}"
)
logger.info(f"QA post-collapse | shape={funda_all.shape}")


# crear fecha mensual a partir del cierre trimestral
funda_all["date"] = funda_all["period_end"].dt.to_period("M").dt.to_timestamp("M")

cols_keep = [
    "ric",
    "date",
    "market_share",
    "industry_sales",
    "revenue",
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]

funda_monthly = funda_all[cols_keep].copy()

# expansión mensual segura por ric
funda_monthly = funda_monthly.sort_values(["ric", "date"])

funda_monthly = (
    funda_monthly
    .groupby("ric", group_keys=False)
    .apply(lambda df: df.set_index("date").resample("M").ffill())
    .reset_index()
)

logger.info(f"QA funda_monthly | shape={funda_monthly.shape}")
logger.info(f"QA funda_monthly | NA market_share={funda_monthly['market_share'].isna().mean():.4f}")


# ==========================================================
# 8) export final (mismo nombre que antes)
# ==========================================================

# panel completo sp500
out_sp500 = OUT.get(
    "fundamentals_with_market_share_sp500",
    RAW_DIR / "fundamentals_with_market_share_sp500.csv",
)

funda_monthly.to_csv(out_sp500, index=False)
logger.info(f"Output -> {out_sp500.as_posix()} | filas={len(funda_monthly):,}")

# panel filtrado a la muestra
funda_monthly["ric_base"] = funda_monthly["ric"].astype(str).str.split(".").str[0]

funda_focus = funda_monthly[funda_monthly["ric_base"].isin(target_bases)].copy()

out_focus = OUT.get(
    "fundamentals_with_market_share_focus",
    RAW_DIR / "fundamentals_with_market_share.csv",
)

funda_focus.to_csv(out_focus, index=False)
logger.info(f"Output -> {out_focus.as_posix()} | filas={len(funda_focus):,}")

### Construcción de proxies alternativas de poder de mercado y competencia

En esta sección se construyen proxies alternativas de posicionamiento relativo y concentración dentro del universo S&P 500, utilizando información de ventas y clasificación GICS con distintos niveles de desagregación. El objetivo es complementar la medida base de participación de ventas y generar variantes más robustas para análisis principal y ejercicios de robustez.

#### Motivación

La medida inicial de `market_share` se construía con un denominador acotado y, por lo tanto, su interpretación como proxy de poder de mercado agregado era limitada. Para mejorar esta aproximación, aquí se amplía el universo comparativo al conjunto de firmas del S&P 500 y se aprovecha la mayor desagregación de la clasificación GICS.

#### Objetivo

Construir un conjunto de proxies comparables de participación relativa, tamaño relativo y concentración, definidas dentro del universo S&P 500 y a distintos niveles de agregación sectorial.

#### Proxies construidas

A partir de ventas trimestrales (`revenue`) y grupos GICS, se generan las siguientes variables:

1. `market_share_industry_group_sp500`: participación de ventas dentro de `GICS Industry Group`.
2. `market_share_industry_sp500`: participación de ventas dentro de `GICS Industry`.
3. `market_share_subindustry_sp500`: participación de ventas dentro de `GICS Sub-Industry`.
4. `hhi_industry_group_sp500`, `hhi_industry_sp500` y `hhi_subindustry_sp500`: índices de concentración medidos como suma de participaciones al cuadrado.
5. `relative_size_industry_group_sp500`, `relative_size_industry_sp500` y `relative_size_subindustry_sp500`: tamaño relativo de la firma respecto de la mediana de ventas del grupo correspondiente.
6. Rankings de ventas dentro de cada grupo y trimestre.

#### Procedimiento

Primero se depura la base trimestral de ventas para asegurar unicidad a nivel (`ric`, `period_end`) y consistencia de la clasificación GICS. Luego, para cada nivel de agregación disponible, se calcula:

- ventas totales del grupo;
- mediana de ventas del grupo;
- cantidad de firmas activas;
- participación de ventas de cada firma;
- tamaño relativo frente a la mediana;
- ranking de ventas;
- índice de concentración HHI.

Posteriormente, la base trimestral se expande a frecuencia mensual mediante forward fill, para facilitar su integración con el panel maestro mensual.

#### Estrategia empírica sugerida

Dentro de este conjunto, una estrategia razonable es utilizar:

- **principal:** `market_share_industry_sp500`;
- **robustez 1:** `market_share_subindustry_sp500`;
- **robustez 2:** `hhi_industry_sp500`;
- **robustez 3:** `relative_size_industry_sp500`.

#### Control de calidad

Se verifican:
- disponibilidad de columnas GICS finas;
- duplicados a nivel firma-trimestre antes del cálculo;
- cobertura de cada proxy luego de la expansión mensual;
- proporción de valores faltantes por variable.

#### Output

Se generan dos archivos de salida:

- `market_power_proxies_sp500_quarterly.csv`: universo completo del S&P 500;
- `market_power_proxies_focus_quarterly.csv`: muestra restringida a la tesis.

Aunque el nombre histórico del archivo se preserva por compatibilidad, la base final exportada en esta etapa queda expresada en frecuencia mensual.

In [ ]:
# ======================
# market power proxies (sp500)
# ======================

import numpy as np
import pandas as pd

# -------------------------------------------------
# 0) base de trabajo
# -------------------------------------------------
req_cols = [
    "ric",
    "period_end",
    "revenue",
    "gics_sector_name",
    "gics_industry_group_name",
]

missing_req = [c for c in req_cols if c not in funda_all.columns]
if missing_req:
    raise RuntimeError(f"Faltan columnas en funda_all: {missing_req}")

logger.info("Cobertura columnas GICS finas en funda_all:")
for c in ["gics_industry_name", "gics_subindustry_name"]:
    logger.info(f"{c}: {c in funda_all.columns}")

mp = funda_all.copy()
mp["period_end"] = pd.to_datetime(mp["period_end"], errors="coerce")
mp["revenue"] = pd.to_numeric(mp["revenue"], errors="coerce")
mp = mp.dropna(subset=["ric", "period_end"]).copy()

# -------------------------------------------------
# 1) colapsar duplicados a nivel firma-trimestre
# -------------------------------------------------
agg_dict = {
    "revenue": "mean",
    "gics_sector_name": "first",
    "gics_industry_group_name": "first",
}

if "gics_industry_name" in mp.columns:
    agg_dict["gics_industry_name"] = "first"

if "gics_subindustry_name" in mp.columns:
    agg_dict["gics_subindustry_name"] = "first"

mp = (
    mp
    .sort_values(["ric", "period_end"])
    .groupby(["ric", "period_end"], as_index=False)
    .agg(agg_dict)
)

logger.info(
    f"QA mp post-collapse | duplicados ric-period_end="
    f"{mp.duplicated(['ric', 'period_end']).sum():,}"
)

# limpieza robusta de strings
for c in [
    "gics_sector_name",
    "gics_industry_group_name",
    "gics_industry_name",
    "gics_subindustry_name",
]:
    if c in mp.columns:
        mp[c] = mp[c].astype("string").str.strip()

# ingresos no positivos se tratan como faltantes
mp.loc[mp["revenue"] <= 0, "revenue"] = np.nan


# -------------------------------------------------
# 2) helper para shares, tamaño relativo y hhi
# -------------------------------------------------
def build_share_hhi(df, group_col, suffix):
    # construye métricas de participación y concentración por grupo GICS
    tmp = df[["ric", "period_end", "revenue", group_col]].copy()
    tmp = tmp.dropna(subset=["period_end", group_col])

    tmp_valid = tmp.dropna(subset=["revenue"]).copy()

    grp = (
        tmp_valid
        .groupby([group_col, "period_end"], as_index=False)
        .agg(
            group_sales=("revenue", "sum"),
            group_median_sales=("revenue", "median"),
            n_firms=("ric", "nunique"),
        )
    )

    grp = grp[grp["group_sales"] > 0]

    out = tmp.merge(grp, on=[group_col, "period_end"], how="left")

    out[f"market_share_{suffix}"] = out["revenue"] / out["group_sales"]
    out[f"relative_size_{suffix}"] = out["revenue"] / out["group_median_sales"]

    out[f"rank_revenue_{suffix}"] = (
        out.groupby([group_col, "period_end"])["revenue"]
        .rank(method="dense", ascending=False)
    )

    hhi = (
        out.dropna(subset=[f"market_share_{suffix}"])
        .assign(share_sq=lambda x: x[f"market_share_{suffix}"] ** 2)
        .groupby([group_col, "period_end"], as_index=False)["share_sq"]
        .sum()
        .rename(columns={"share_sq": f"hhi_{suffix}"})
    )

    out = out.merge(hhi, on=[group_col, "period_end"], how="left")

    out = out.rename(columns={
        "group_sales": f"group_sales_{suffix}",
        "group_median_sales": f"group_median_sales_{suffix}",
        "n_firms": f"n_firms_{suffix}",
    })

    keep = [
        "ric",
        "period_end",
        f"group_sales_{suffix}",
        f"group_median_sales_{suffix}",
        f"n_firms_{suffix}",
        f"market_share_{suffix}",
        f"relative_size_{suffix}",
        f"rank_revenue_{suffix}",
        f"hhi_{suffix}",
    ]

    return out[keep].drop_duplicates(subset=["ric", "period_end"])


# -------------------------------------------------
# 3) construcción de proxies por nivel GICS
# -------------------------------------------------
logger.info("Construyendo proxies de market power...")

proxies = []

# industry group siempre debería existir
proxies.append(
    build_share_hhi(mp, "gics_industry_group_name", "industry_group_sp500")
)

# industry y subindustry son opcionales según cobertura
if "gics_industry_name" in mp.columns:
    proxies.append(
        build_share_hhi(mp, "gics_industry_name", "industry_sp500")
    )

if "gics_subindustry_name" in mp.columns:
    proxies.append(
        build_share_hhi(mp, "gics_subindustry_name", "subindustry_sp500")
    )


# -------------------------------------------------
# 4) merge horizontal de todas las proxies
# -------------------------------------------------
base_cols = [
    c for c in [
        "ric",
        "period_end",
        "revenue",
        "gics_sector_name",
        "gics_industry_group_name",
        "gics_industry_name",
        "gics_subindustry_name",
    ]
    if c in mp.columns
]

market_power = mp[base_cols].copy()

for piece in proxies:
    market_power = market_power.merge(
        piece,
        on=["ric", "period_end"],
        how="left",
    )


# -------------------------------------------------
# 5) variables adicionales
# -------------------------------------------------
market_power["log_revenue"] = np.log(market_power["revenue"])

if "n_firms_subindustry_sp500" in market_power.columns:
    market_power["small_subindustry_group"] = (
        market_power["n_firms_subindustry_sp500"].fillna(0) < 4
    ).astype(int)

if "n_firms_industry_sp500" in market_power.columns:
    market_power["small_industry_group"] = (
        market_power["n_firms_industry_sp500"].fillna(0) < 4
    ).astype(int)

market_power["ric_base"] = market_power["ric"].astype(str).str.split(".").str[0]


# -------------------------------------------------
# 6) quarterly -> monthly
# -------------------------------------------------
market_power = market_power.sort_values(["ric", "period_end"])
market_power["date"] = market_power["period_end"].dt.to_period("M").dt.to_timestamp("M")

market_power = (
    market_power
    .groupby("ric", group_keys=False)
    .apply(lambda df: df.set_index("date").resample("M").ffill())
    .reset_index()
)

logger.info(
    f"QA market_power monthly | NA industry_group_share="
    f"{market_power['market_share_industry_group_sp500'].isna().mean():.4f}"
)

if "market_share_industry_sp500" in market_power.columns:
    logger.info(
        f"QA market_power monthly | NA industry_share="
        f"{market_power['market_share_industry_sp500'].isna().mean():.4f}"
    )

if "market_share_subindustry_sp500" in market_power.columns:
    logger.info(
        f"QA market_power monthly | NA subindustry_share="
        f"{market_power['market_share_subindustry_sp500'].isna().mean():.4f}"
    )


# -------------------------------------------------
# 7) export
# -------------------------------------------------
out_all = OUT.get(
    "market_power_proxies_sp500_q",
    RAW_DIR / "market_power_proxies_sp500_quarterly.csv",
)
market_power.to_csv(out_all, index=False)

market_power_focus = market_power[market_power["ric_base"].isin(target_bases)].copy()

out_focus = OUT.get(
    "market_power_proxies_focus_q",
    RAW_DIR / "market_power_proxies_focus_quarterly.csv",
)
market_power_focus.to_csv(out_focus, index=False)

logger.info(f"Output -> {out_all.as_posix()} | filas={len(market_power):,}")
logger.info(f"Output -> {out_focus.as_posix()} | filas={len(market_power_focus):,}")


# -------------------------------------------------
# 8) cobertura final
# -------------------------------------------------
summary_cols = [
    c for c in [
        "market_share_industry_group_sp500",
        "market_share_industry_sp500",
        "market_share_subindustry_sp500",
        "hhi_industry_group_sp500",
        "hhi_industry_sp500",
        "hhi_subindustry_sp500",
    ]
    if c in market_power_focus.columns
]

coverage = (
    market_power_focus[summary_cols]
    .notna()
    .mean()
    .sort_values(ascending=False)
)

logger.info("Cobertura proxies market power:")
logger.info(coverage.to_string())

### Validación de consistencia de las proxies de market power

En esta sección se realizan controles de calidad específicos sobre las variables de poder de mercado construidas a partir de ventas agregadas dentro del universo del S&P 500. El objetivo es verificar que las proxies de concentración y tamaño relativo sean internamente consistentes y que su interpretación económica no esté dominada por problemas de cobertura o de estructura del panel.

#### Objetivo

Evaluar la robustez descriptiva de las medidas de market power a partir de tres verificaciones complementarias:

1. la distribución del número de firmas por grupo de comparación;
2. la identificación de casos extremos en los que el índice de concentración toma valor uno;
3. la unicidad de las observaciones a nivel firma-período.

#### 1. Distribución del número de firmas por grupo

Se analiza la distribución del número de firmas presentes en cada nivel de agregación sectorial disponible (`industry group`, `industry` y `subindustry`). Este control es relevante porque las medidas de concentración dependen mecánicamente del tamaño efectivo del grupo sobre el cual se calculan.

En particular, el índice de Herfindahl-Hirschman se define como:

$$
HHI_{g,t} = \sum_{i \in g} s_{i,g,t}^{2}
$$

donde $s_{i,g,t}$ representa la participación relativa de la firma $i$ dentro del grupo $g$ en el período $t$.

Cuando el número de firmas en un grupo es muy reducido, pueden surgir valores extremos de concentración que no necesariamente reflejan poder de mercado en sentido económico, sino una definición estrecha del universo de comparación dentro del S&P 500.

#### 2. Identificación de casos extremos: $HHI = 1$

Se detectan observaciones en las cuales el índice de concentración toma su valor máximo:

$$
HHI_{g,t} = 1
$$

Este resultado implica que una única firma explica el 100% de las ventas observadas en el grupo definido. En este contexto empírico, estos casos pueden surgir por al menos tres motivos:

- existencia de una única firma del S&P 500 en esa categoría;
- cobertura incompleta del mercado relevante al restringir el análisis al universo del índice;
- problemas de construcción o agregación en la base de datos.

Por ello, estos casos se inspeccionan explícitamente antes de utilizar las proxies en el análisis econométrico.

#### 3. Consistencia del panel a nivel firma-tiempo

Finalmente, se verifica la existencia de duplicados en la combinación identificadora `ric-period_end`. La unicidad de esta dimensión es una condición básica para la correcta construcción del panel mensual y para evitar sesgos en etapas posteriores de merge, agregación o estimación.

#### Alcance interpretativo

Dado que las medidas se construyen dentro del universo S&P 500, estas proxies deben interpretarse principalmente como indicadores de **posición relativa entre grandes firmas** y no como una medida exhaustiva de poder de mercado en el mercado total.

In [ ]:
# ==========================================================
# validación descriptiva de proxies de market power
# ==========================================================

# ----------------------------------------------------------
# 1) distribución del número de firmas por grupo
# ----------------------------------------------------------
cols_n_firms = [
    "n_firms_industry_group_sp500",
    "n_firms_industry_sp500",
    "n_firms_subindustry_sp500",
]

for c in cols_n_firms:
    if c in market_power_focus.columns:
        logger.info(f"\nDistribución de {c}")
        logger.info(market_power_focus[c].describe())
        logger.info("Frecuencias (primeros 10 valores ordenados):")
        logger.info(market_power_focus[c].value_counts().sort_index().head(10))
    else:
        logger.warning(f"{c} -> NO EXISTE en market_power_focus")


# ----------------------------------------------------------
# 2) casos extremos donde hhi = 1
# ----------------------------------------------------------
cols_hhi = [
    "hhi_industry_group_sp500",
    "hhi_industry_sp500",
    "hhi_subindustry_sp500",
]

for c in cols_hhi:
    if c not in market_power.columns:
        logger.warning(f"{c} -> NO EXISTE en market_power")
        continue

    logger.info(f"\nCasos con {c} = 1")

    cols_show = ["ric", "period_end", c]

    extra_cols = [
        "gics_industry_group_name",
        "gics_industry_name",
        "gics_subindustry_name",
        "n_firms_industry_group_sp500",
        "n_firms_industry_sp500",
        "n_firms_subindustry_sp500",
    ]

    cols_show += [col for col in extra_cols if col in market_power.columns]

    hhi_eq_1 = market_power.loc[market_power[c] == 1, cols_show].copy()

    logger.info(hhi_eq_1.head(20))
    logger.info(f"Total de casos con {c} = 1: {(market_power[c] == 1).sum()}")


# ----------------------------------------------------------
# 3) verificación de duplicados ric-period_end
# ----------------------------------------------------------
dup_check = (
    mp.groupby(["ric", "period_end"])
      .size()
      .reset_index(name="n")
      .query("n > 1")
)

logger.info(f"\nDuplicados ric-period_end: {len(dup_check)}")
logger.info(dup_check.head(20))

## Controles macroeconómicos agregados — descarga y construcción mensual

### Objetivo

Incorporar un conjunto parsimonioso de controles macroeconómicos agregados para especificaciones sin efectos fijos de tiempo, con el fin de evaluar la robustez de los coeficientes asociados a shocks agregados de mercado y crédito.

La inclusión de estos controles permite aislar, en la medida de lo posible, la variación atribuible a condiciones macroeconómicas generales de aquella asociada a shocks financieros específicos.

---

### Criterio de selección

Se priorizan variables con interpretación económica directa y bajo solapamiento conceptual con los factores estructurales ya incorporados en el modelo.

**Set base:**
- `policy_rate`: tasa de política monetaria de corto plazo.
- `term_spread_10y_2y`: pendiente de la curva del Tesoro.
- `indpro_yoy`: variación interanual de la actividad económica.

**Set ampliado (robustez):**
- `cpi_yoy`: inflación interanual.
- `vix_eom`: incertidumbre agregada en equity.
- `move_eom`: volatilidad implícita en Treasuries.

---

### Construcción

La pendiente de la curva se define como:

$$
term\_spread_{t} = y^{10y}_{t} - y^{2y}_{t}
$$

donde $y^{10y}_{t}$ y $y^{2y}_{t}$ corresponden a los rendimientos del Tesoro a 10 y 2 años, respectivamente.

En el caso de la actividad económica, si la serie se descarga en niveles, se construye la variación interanual como:

$$
indpro\_yoy_{t} = 100 \times \left( \frac{IP_{t}}{IP_{t-12}} - 1 \right)
$$

---

### Metodología de frecuencia

- Las series diarias se agregan a frecuencia mensual utilizando el **último dato disponible del mes**.
- Las series ya mensuales se alinean directamente a fin de mes.
- Todas las variables se expresan en frecuencia mensual para permitir el merge consistente con el panel firma–mes.

---

### Estrategia de implementación

Debido a posibles restricciones de acceso en Refinitiv Workspace, se implementa un esquema robusto de descarga:

- Para cada variable se define un conjunto de RICs candidatos.
- Se intenta la descarga secuencialmente hasta encontrar una serie válida.
- Se registra el RIC efectivamente utilizado para garantizar trazabilidad.
- Si no se encuentra una serie directa para `policy_rate`, se utiliza como proxy el rendimiento del Tesoro a 3 meses.

---

### Salidas

- `data/raw/macro_controls_daily.parquet`
- `data/clean/macro_controls_monthly.parquet`
- `data/clean/macro_controls_sources.csv`

---

### Validación operativa

La celda incluye controles automáticos de:

- cobertura de cada serie (valores no nulos),
- consistencia temporal,
- identificación de proxies utilizadas,
- y verificación de transformación correcta de variables.

Esto asegura que las variables macroeconómicas sean comparables en el tiempo y consistentes con el resto del pipeline empírico.

---

### Alcance interpretativo

Dado que las especificaciones principales incluyen efectos fijos de tiempo, estas variables se utilizan principalmente en:

- modelos sin efectos fijos temporales,
- ejercicios de robustez,
- y validaciones de sensibilidad de los coeficientes estructurales.

En consecuencia, su rol es complementario y no sustitutivo de la identificación basada en shocks agregados explícitos.

In [ ]:
# ==========================================================
# 16) CONTROLES MACRO -> descarga + construcción mensual
# ==========================================================

# Requiere:
# - START_DATE, END_DATE
# - RAW_DIR, CLEAN_DIR, OUT
# - logger, sleep_request
# - get_ts_safe(...)
# - ust_yield_curve_daily.parquet ya generado


# ==========================================================
# 16.1 CONFIGURACIÓN
# ==========================================================

# Variables:
# baseline: policy_rate, term_spread_10y_2y, indpro_yoy
# robustez: cpi_yoy, vix_eom, move_eom

# Nota:
# TED spread no se incluye por quiebre estructural post-LIBOR

MACRO_RIC_CANDIDATES = {
    # incertidumbre
    "vix": [".VIX", "VIX"],
    "move": [".MOVE", "MOVE"],

    # política monetaria
    "policy_rate": ["USFFTARGET=", "USFEDFUNDS=", "FEDFUNDS"],

    # inflación
    "cpi_yoy": ["USCPIY=ECI", "aUSCPIY", "USCPNY=ECI"],

    # actividad real
    "indpro_yoy": ["USIPY=ECI", "aUSIPY", "USIP=ECI"],
}

MACRO_FREQ = {
    "vix": "daily",
    "move": "daily",
    "policy_rate": "monthly",
    "cpi_yoy": "monthly",
    "indpro_yoy": "monthly",
}


# ==========================================================
# 16.2 DESCARGA DE SERIES MACRO
# ==========================================================

logger.info("Descargando controles macro...")

macro_long = []
macro_sources = []

for var_name, ric_list in MACRO_RIC_CANDIDATES.items():

    interval = MACRO_FREQ[var_name]

    df_var, ric_used = fetch_first_valid_series(
        var_name=var_name,
        candidates=ric_list,
        start_date=START_DATE,
        end_date=END_DATE,
        interval=interval,
        max_attempts=RETRY_MAX,
    )

    if not df_var.empty:
        macro_long.append(df_var)

    macro_sources.append({
        "variable": var_name,
        "ric_used": ric_used,
        "download_status": "ok" if ric_used else "missing",
        "frequency_requested": interval,
    })

    sleep_request(0.5)

macro_sources = pd.DataFrame(macro_sources)

macro_raw = (
    pd.concat(macro_long, ignore_index=True)
    if macro_long else
    pd.DataFrame(columns=["date", "value", "variable", "ric"])
)

# export raw
safe_write_parquet(
    macro_raw,
    OUT.get("macro_controls_daily", RAW_DIR / "macro_controls_daily.parquet"),
    index=False
)


# ==========================================================
# 16.3 CONSTRUCCIÓN DE CURVA UST MENSUAL
# ==========================================================

ust = pd.read_parquet(
    OUT.get("ust_yield_curve_daily", RAW_DIR / "ust_yield_curve_daily.parquet")
).copy()

ust["date"] = pd.to_datetime(ust["date"], errors="coerce")

ust_m = (
    ust.dropna(subset=["date", "tenor", "yield"])
       .sort_values(["tenor", "date"])
       .assign(date=lambda x: x["date"].dt.to_period("M").dt.to_timestamp("M"))
       .groupby(["date", "tenor"], as_index=False)["yield"].last()
       .pivot(index="date", columns="tenor", values="yield")
       .reset_index()
)

ust_m = ust_m.rename(columns={
    "US3M": "ust_3m",
    "US2Y": "ust_2y",
    "US10Y": "ust_10y",
})


# ==========================================================
# 16.4 CONSTRUCCIÓN DE SPREADS
# ==========================================================

if {"ust_10y", "ust_2y"}.issubset(ust_m.columns):
    ust_m["term_spread_10y_2y"] = ust_m["ust_10y"] - ust_m["ust_2y"]

if {"ust_10y", "ust_3m"}.issubset(ust_m.columns):
    ust_m["term_spread_10y_3m"] = ust_m["ust_10y"] - ust_m["ust_3m"]


# ==========================================================
# 16.5 AGREGACIÓN A FRECUENCIA MENSUAL
# ==========================================================

monthly_parts = [ust_m.copy()]

if not macro_raw.empty:

    def add_if_exists(var, out_name):
        if (macro_raw["variable"] == var).any():
            monthly_parts.append(
                monthly_last(
                    macro_raw.loc[macro_raw["variable"] == var],
                    "date",
                    "value",
                    out_name
                )
            )

    add_if_exists("vix", "vix_eom")
    add_if_exists("move", "move_eom")
    add_if_exists("policy_rate", "policy_rate")
    add_if_exists("cpi_yoy", "cpi_yoy")
    add_if_exists("indpro_yoy", "indpro_yoy")


# merge
macro_monthly = None
for part in monthly_parts:
    macro_monthly = part if macro_monthly is None else macro_monthly.merge(part, on="date", how="outer")

macro_monthly = macro_monthly.sort_values("date").reset_index(drop=True)


# ==========================================================
# 16.6 FALLBACK POLICY RATE
# ==========================================================

if "policy_rate" not in macro_monthly or macro_monthly["policy_rate"].notna().sum() == 0:

    if "ust_3m" in macro_monthly:
        macro_monthly["policy_rate"] = macro_monthly["ust_3m"]
        logger.warning("policy_rate no disponible -> usando proxy ust_3m")


# ==========================================================
# 16.7 TRANSFORMACIÓN INDPRO SI VIENE EN NIVEL
# ==========================================================

if "indpro_yoy" in macro_monthly:
    s = pd.to_numeric(macro_monthly["indpro_yoy"], errors="coerce")

    if s.notna().sum() >= 24 and s.median() > 20:
        macro_monthly = macro_monthly.sort_values("date")
        macro_monthly["indpro_yoy"] = 100 * (
            macro_monthly["indpro_yoy"] /
            macro_monthly["indpro_yoy"].shift(12) - 1
        )
        logger.info("indpro transformado a YoY")


# ==========================================================
# 16.8 EXPORT FINAL
# ==========================================================

safe_write_parquet(
    macro_monthly,
    OUT.get("macro_controls_monthly", CLEAN_DIR / "macro_controls_monthly.parquet"),
    index=False
)

macro_sources.to_csv(CLEAN_DIR / "macro_controls_sources.csv", index=False)


# ==========================================================
# 16.9 RESUMEN
# ==========================================================

logger.info("Columnas macro:")
logger.info(macro_monthly.columns.tolist())

logger.info("Cobertura:")
logger.info(macro_monthly.notna().sum().sort_values(ascending=False))

## 17) Outputs documentales para la Sección Datos — universo, fuentes y cobertura bruta

### Objetivo

Esta sección construye un conjunto de outputs documentales destinados a describir de forma transparente el universo empírico de la tesis, las fuentes utilizadas y la cobertura efectiva de los datos.

A diferencia del pipeline principal, estas celdas no forman parte del ETL ni intervienen en la construcción del panel final. Su propósito es exclusivamente **documental**, facilitando la redacción de la Sección 3 (Datos) y asegurando la trazabilidad del proceso empírico.

---

### Estructura de la sección

La sección se organiza en tres bloques complementarios:

#### 17A — Configuración y funciones auxiliares

Se definen directorios de salida y funciones helper que permiten:

- cargar automáticamente el primer archivo disponible entre múltiples alternativas;
- identificar columnas equivalentes bajo distintos nombres (e.g. `RIC`, `ticker`, `issuer`);
- estandarizar fechas;
- construir resúmenes estructurados de cada dataset.

Estas funciones permiten trabajar con datasets heterogéneos sin introducir supuestos rígidos sobre nombres de columnas o formatos.

---

#### 17B — Resumen del universo y cobertura de datasets

Se construyen tablas que sintetizan, para cada dataset relevante:

- existencia del archivo;
- tamaño (filas y columnas);
- número de firmas;
- número de bonos;
- número de tickers;
- rango temporal observado.

Este bloque permite evaluar rápidamente la cobertura relativa de cada fuente y detectar posibles limitaciones de datos.

---

#### 17C — Outputs específicos para redacción

Se generan tablas finales listas para ser utilizadas en la redacción de la tesis, incluyendo:

- universo bruto de bonos;
- universo elegible tras filtros;
- cobertura de variables clave (e.g. OAS);
- consistencia temporal del panel.

---

### Enfoque metodológico

Dado que los datasets provienen de distintas fuentes y APIs, se adopta un enfoque robusto basado en:

- detección flexible de columnas equivalentes;
- carga tolerante a múltiples formatos (CSV / Parquet);
- construcción automática de métricas descriptivas.

Esto permite asegurar reproducibilidad sin depender de estructuras rígidas de datos.

---

### Alcance interpretativo

Estos outputs documentan el **universo observado de datos**, no necesariamente el universo económico completo.

En particular:

- la cobertura refleja disponibilidad en Refinitiv, no totalidad del mercado;
- los conteos dependen de identificadores observables (RIC, ticker, etc.);
- las métricas deben interpretarse como descriptivas y no como estadísticas estructurales.

No constituyen resultados econométricos, pero son fundamentales para justificar:

- la construcción del panel,
- las restricciones muestrales,
- y la validez empírica del análisis.

In [ ]:
# ==========================================================
# 17A CONFIG + HELPERS | OUTPUTS DOCUMENTALES SECCIÓN DATOS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ==========================================================
# 17A.1 DIRECTORIOS DE SALIDA
# ==========================================================

DATASEC_DIR = OUTPUT_DIR / "tables" / "data_section"
DATASEC_QA_DIR = QA_DIR / "data_section"

for p in [DATASEC_DIR, DATASEC_QA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

logger.info(f"Directorio documental: {DATASEC_DIR.as_posix()}")


# ==========================================================
# 17A.2 HELPERS DE CARGA
# ==========================================================

def load_first_existing(paths):
    """
    Devuelve (df, path) del primer archivo existente.
    Si no encuentra ninguno, devuelve (None, None).
    """
    for p in paths:
        if p is None:
            continue

        p = Path(p)

        if p.exists():
            if p.suffix == ".parquet":
                return pd.read_parquet(p), p

            if p.suffix == ".csv":
                return pd.read_csv(p), p

    return None, None


def first_existing_col(df: pd.DataFrame, candidates: list[str]):
    """
    Devuelve el primer nombre de columna existente dentro de una lista
    de candidatos equivalentes, ignorando mayúsculas/minúsculas y espacios.
    """
    cols = {str(c).strip().lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]

    return None


def to_datetime_safe(s):
    """
    Convierte una serie a datetime de forma robusta.
    """
    return pd.to_datetime(s, errors="coerce")


# ==========================================================
# 17A.3 HELPER DE RESUMEN DOCUMENTAL
# ==========================================================

def dataset_summary(name: str, df: pd.DataFrame, path: Path):
    """
    Construye un resumen documental estandarizado de un dataset.
    """

    if df is None:
        return {
            "dataset": name,
            "path": None,
            "exists": 0,
            "n_rows": np.nan,
            "n_cols": np.nan,
            "n_firms": np.nan,
            "n_bonds": np.nan,
            "n_tickers": np.nan,
            "date_min": pd.NaT,
            "date_max": pd.NaT,
        }

    firm_col = first_existing_col(
        df,
        ["issuer", "Issuer", "ric", "RIC", "ticker", "Ticker"]
    )

    bond_col = first_existing_col(
        df,
        ["bond_ric", "Preferred RIC", "RIC", "Instrument"]
    )

    ticker_col = first_existing_col(
        df,
        ["ticker", "Ticker"]
    )

    date_col = first_existing_col(
        df,
        ["date", "Date", "period_end", "Period End", "TR.OPTIONADJUSTEDSPREADBID Date"]
    )

    n_firms = df[firm_col].nunique(dropna=True) if firm_col else np.nan
    n_bonds = df[bond_col].nunique(dropna=True) if bond_col else np.nan
    n_tickers = df[ticker_col].nunique(dropna=True) if ticker_col else np.nan

    if date_col:
        d = to_datetime_safe(df[date_col])
        date_min = d.min()
        date_max = d.max()
    else:
        date_min, date_max = pd.NaT, pd.NaT

    return {
        "dataset": name,
        "path": str(path) if path is not None else None,
        "exists": 1,
        "n_rows": len(df),
        "n_cols": df.shape[1],
        "n_firms": n_firms,
        "n_bonds": n_bonds,
        "n_tickers": n_tickers,
        "date_min": date_min,
        "date_max": date_max,
    }

In [ ]:
# ==========================================================
# 17B INVENTARIO DE DATASETS / FUENTES DESCARGADAS
# ==========================================================

# ----------------------------------------------------------
# 17B.1 DEFINICIÓN DE DATASETS A AUDITAR
# ----------------------------------------------------------

datasets_to_audit = {

    # universo
    "universe_input_csv": [
        UNIVERSE_FILE
    ],

    "universe_firms_clean": [
        OUT.get("universe_firms"),
        Path(str(OUT.get("universe_firms"))).with_suffix(".csv") if OUT.get("universe_firms") else None
    ],

    # metadata
    "company_metadata": [
        RAW_DIR / "company_metadata.parquet",
        RAW_DIR / "company_metadata.csv"
    ],

    # bonos
    "bonds_universe_full": [
        OUT.get("bonds_universe_full"),
        Path(str(OUT.get("bonds_universe_full"))).with_suffix(".csv") if OUT.get("bonds_universe_full") else None
    ],

    "oas_spreads_monthly_bond": [
        OUT.get("oas_spreads_monthly_bond"),
        Path(str(OUT.get("oas_spreads_monthly_bond"))).with_suffix(".csv") if OUT.get("oas_spreads_monthly_bond") else None
    ],

    "bonds_monthly_spreads_clean": [
        CLEAN_DIR / "bonds_monthly_spreads.parquet",
        CLEAN_DIR / "bonds_monthly_spreads.csv"
    ],

    # equity
    "equity_prices_daily": [
        RAW_DIR / "equity_prices_daily.parquet",
        RAW_DIR / "equity_prices_daily.csv"
    ],

    "equity_returns_daily": [
        CLEAN_DIR / "equity_returns_daily.parquet",
        CLEAN_DIR / "equity_returns_daily.csv"
    ],

    # mercado
    "market_prices_daily": [
        RAW_DIR / "market_prices_daily.parquet",
        RAW_DIR / "market_prices_daily.csv",
        RAW_DIR / "market_sp500_daily.parquet",
        RAW_DIR / "market_sp500_daily.csv"
    ],

    # fundamentals
    "fundamentals_q_extended": [
        RAW_DIR / "fundamentals_extended_q.parquet",
        RAW_DIR / "fundamentals_extended_q.csv"
    ],

    "fundamentals_with_market_share": [
        RAW_DIR / "fundamentals_with_market_share_sp500.csv",
        RAW_DIR / "fundamentals_with_market_share.csv"
    ],

    # market power
    "market_power_proxies": [
        RAW_DIR / "market_power_proxies_sp500_quarterly.csv",
        RAW_DIR / "market_power_proxies_focus_quarterly.csv"
    ],

    # cds
    "cds_spreads_daily": [
        OUT.get("cds_spreads_daily"),
        Path(str(OUT.get("cds_spreads_daily"))).with_suffix(".csv") if OUT.get("cds_spreads_daily") else None
    ],

    # macro
    "macro_controls_monthly": [
        CLEAN_DIR / "macro_controls_monthly.parquet",
        CLEAN_DIR / "macro_controls_monthly.csv"
    ],
}


# ----------------------------------------------------------
# 17B.2 CONSTRUCCIÓN DEL INVENTARIO
# ----------------------------------------------------------

inventory_rows = []
loaded_cache = {}

for name, candidates in datasets_to_audit.items():

    df, path = load_first_existing(candidates)

    loaded_cache[name] = (df, path)

    inventory_rows.append(
        dataset_summary(name, df, path)
    )


# ----------------------------------------------------------
# 17B.3 EXPORT
# ----------------------------------------------------------

inventory_df = (
    pd.DataFrame(inventory_rows)
    .sort_values(["exists", "dataset"], ascending=[False, True])
)

inventory_path = DATASEC_DIR / "01_download_inventory.csv"

inventory_df.to_csv(inventory_path, index=False)

print(f"Inventario guardado en: {inventory_path.as_posix()}")
display(inventory_df)

In [ ]:
# ==========================================================
# 17C TABLAS DOCUMENTALES: UNIVERSO, BONOS Y COBERTURA OAS
# ==========================================================

summary_rows = []


# ----------------------------------------------------------
# 17C.1 UNIVERSO INICIAL
# ----------------------------------------------------------

df_univ, _ = loaded_cache.get("universe_input_csv", (None, None))

if df_univ is not None:

    firm_col = first_existing_col(df_univ, ["ticker", "Ticker", "ric", "RIC", "issuer", "Issuer"])

    summary_rows.append({
        "stage": "01_universe_input",
        "n_rows": len(df_univ),
        "n_firms": df_univ[firm_col].nunique(dropna=True) if firm_col else np.nan,
        "n_bonds": np.nan,
        "n_obs": len(df_univ),
        "notes": "Universo inicial de firmas / tickers",
    })


# ----------------------------------------------------------
# 17C.2 METADATA CORPORATIVA
# ----------------------------------------------------------

df_meta, _ = loaded_cache.get("company_metadata", (None, None))

if df_meta is not None:

    firm_col = first_existing_col(df_meta, ["ric", "RIC", "ticker", "Ticker", "issuer", "Issuer"])

    summary_rows.append({
        "stage": "02_company_metadata",
        "n_rows": len(df_meta),
        "n_firms": df_meta[firm_col].nunique(dropna=True) if firm_col else np.nan,
        "n_bonds": np.nan,
        "n_obs": len(df_meta),
        "notes": "Metadata corporativa (Refinitiv)",
    })


# ----------------------------------------------------------
# 17C.3 UNIVERSO DE BONOS FILTRADO
# ----------------------------------------------------------

df_bonds, _ = loaded_cache.get("bonds_universe_full", (None, None))

if df_bonds is not None:

    firm_col = first_existing_col(df_bonds, ["issuer", "Issuer", "ticker", "Ticker"])
    bond_col = first_existing_col(df_bonds, ["bond_ric", "Preferred RIC", "RIC"])

    summary_rows.append({
        "stage": "03_bonds_universe_filtered",
        "n_rows": len(df_bonds),
        "n_firms": df_bonds[firm_col].nunique(dropna=True) if firm_col else np.nan,
        "n_bonds": df_bonds[bond_col].nunique(dropna=True) if bond_col else np.nan,
        "n_obs": len(df_bonds),
        "notes": "Bonos elegibles tras filtros",
    })


# ----------------------------------------------------------
# 17C.4 OAS RAW BONO-MES
# ----------------------------------------------------------

df_oas, _ = loaded_cache.get("oas_spreads_monthly_bond", (None, None))

if df_oas is not None:

    bond_col = first_existing_col(df_oas, ["bond_ric", "Instrument", "RIC"])
    firm_col = first_existing_col(df_oas, ["issuer", "Issuer", "ticker", "Ticker"])
    date_col = first_existing_col(df_oas, ["date", "Date"])

    d = to_datetime_safe(df_oas[date_col]) if date_col else pd.Series(dtype="datetime64[ns]")

    summary_rows.append({
        "stage": "04_oas_raw_bond_month",
        "n_rows": len(df_oas),
        "n_firms": df_oas[firm_col].nunique(dropna=True) if firm_col else np.nan,
        "n_bonds": df_oas[bond_col].nunique(dropna=True) if bond_col else np.nan,
        "n_obs": len(df_oas),
        "notes": f"OAS bruto bono-mes | rango: {d.min()} -> {d.max()}",
    })


# ----------------------------------------------------------
# 17C.5 BOND-MONTH LIMPIO
# ----------------------------------------------------------

df_bm, _ = loaded_cache.get("bonds_monthly_spreads_clean", (None, None))

if df_bm is not None:

    bond_col = first_existing_col(df_bm, ["bond_ric"])
    firm_col = first_existing_col(df_bm, ["issuer", "Issuer"])

    summary_rows.append({
        "stage": "05_bond_month_clean",
        "n_rows": len(df_bm),
        "n_firms": df_bm[firm_col].nunique(dropna=True) if firm_col else np.nan,
        "n_bonds": df_bm[bond_col].nunique(dropna=True) if bond_col else np.nan,
        "n_obs": len(df_bm),
        "notes": "Bond-month limpio enriquecido",
    })


# ----------------------------------------------------------
# 17C.6 EXPORT FUNNEL
# ----------------------------------------------------------

download_funnel = pd.DataFrame(summary_rows)

download_funnel_path = DATASEC_DIR / "01_download_funnel.csv"

download_funnel.to_csv(download_funnel_path, index=False)

print(f"Download funnel guardado en: {download_funnel_path.as_posix()}")
display(download_funnel)


# ----------------------------------------------------------
# 17C.7 COBERTURA OAS
# ----------------------------------------------------------

oas_cov_file = RAW_DIR / "oas_coverage_report.csv"
oas_fail_file = RAW_DIR / "oas_failed_rics.csv"

if oas_cov_file.exists():

    oas_cov = pd.read_csv(oas_cov_file)

    oas_cov_summary = pd.DataFrame([{
        "bond_rics_total_reported": len(oas_cov),
        "bond_rics_with_oas": int(oas_cov.get("covered", pd.Series(dtype=float)).fillna(0).sum())
            if "covered" in oas_cov.columns else np.nan,
        "coverage_pct": round(
            oas_cov.get("covered", pd.Series(dtype=float)).fillna(0).mean() * 100,
            2
        ) if "covered" in oas_cov.columns else np.nan,
    }])

else:
    oas_cov_summary = pd.DataFrame()


if oas_fail_file.exists():

    oas_fail = pd.read_csv(oas_fail_file)

    if oas_cov_summary.empty:
        oas_cov_summary = pd.DataFrame([{}])

    oas_cov_summary["failed_rics_file_count"] = len(oas_fail)


oas_cov_summary_path = DATASEC_DIR / "01_oas_coverage_summary.csv"

oas_cov_summary.to_csv(oas_cov_summary_path, index=False)

print(f"OAS coverage summary guardado en: {oas_cov_summary_path.as_posix()}")
display(oas_cov_summary)

In [ ]:
# ==========================================================
# 17D REPORTE CORTO EN MARKDOWN
# ==========================================================

inv = inventory_df.copy()
funnel = download_funnel.copy()


def pick_value(df, stage, col):
    x = df.loc[df["stage"] == stage, col]
    return x.iloc[0] if len(x) else "NA"


report_lines = [

    "# Resumen documental — Notebook 01 (descarga de datos)",
    "",

    "## Universo y cobertura bruta",

    f"- Universo inicial: {pick_value(funnel, '01_universe_input', 'n_firms')} firmas/tickers.",
    f"- Metadata corporativa: {pick_value(funnel, '02_company_metadata', 'n_firms')} identificadores con metadata.",
    f"- Universo de bonos filtrado: {pick_value(funnel, '03_bonds_universe_filtered', 'n_bonds')} bond RICs elegibles para {pick_value(funnel, '03_bonds_universe_filtered', 'n_firms')} firmas.",
    f"- OAS bruto descargado: {pick_value(funnel, '04_oas_raw_bond_month', 'n_obs')} observaciones bono-mes.",
    f"- Bond-month limpio enriquecido: {pick_value(funnel, '05_bond_month_clean', 'n_obs')} observaciones.",

    "",

    "## Archivos auxiliares generados",

    f"- Inventario de datasets: `{inventory_path.name}`",
    f"- Embudo de descarga: `{download_funnel_path.name}`",
    f"- Cobertura OAS: `{oas_cov_summary_path.name}`",

    "",

    "## Nota",

    "Estos outputs sirven como insumo directo para la redacción de la Sección 3 (Datos), en particular las subsecciones de universo, fuentes y cobertura.",

]


report_text = "\n".join(report_lines)

report_path = DATASEC_QA_DIR / "01_download_report.md"

report_path.write_text(report_text, encoding="utf-8")


print(report_text)
print(f"\nReporte markdown guardado en: {report_path.as_posix()}")